In [ ]:
# Portable project paths. Set TLS_PROJECT_ROOT to the directory containing the input data.
import os
from pathlib import Path
PROJECT_ROOT = Path(os.environ.get("TLS_PROJECT_ROOT", ".")).resolve()


In [ ]:
import re
import warnings
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
from scipy.sparse import issparse
from scipy.stats import mannwhitneyu, spearmanr, rankdata

warnings.filterwarnings('ignore')

sns.set_theme(context='talk', style='white')
mpl.rcParams.update({
    # ── Adobe Illustrator 兼容：字体嵌入为可编辑文字，而非路径 ──
    'pdf.fonttype': 42,       # Type 42 = TrueType, AI 可直接编辑文字
    'ps.fonttype':  42,
    'svg.fonttype': 'none',   # SVG 同样保持可编辑文字
    # ── 分辨率 ──
    'figure.dpi':   150,
    'savefig.dpi':  300,
    'savefig.bbox': 'tight',
    # ── 背景 ──
    'figure.facecolor': 'white',
    'axes.facecolor':   'white',
    # ── 坐标轴 ──
    'axes.spines.top':    False,
    'axes.spines.right':  False,
    'axes.linewidth':     1.2,
    'axes.titleweight':   'bold',
    # ── 字体族（Arial 在 AI 中显示最佳）──
    'font.family':     'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    # ── 全局字体大小（统一放大）──
    'font.size':          16,
    'axes.titlesize':     20,
    'axes.labelsize':     17,
    'xtick.labelsize':    15,
    'ytick.labelsize':    15,
    'legend.fontsize':    14,
    'legend.title_fontsize': 15,
    'legend.frameon':     False,
})

COLOR_NORMAL    = '#2563EB'
COLOR_DEVIATING = '#DC2626'
COLOR_NEUTRAL   = '#D1D5DB'
COLOR_TEXT      = '#1E293B'
COLOR_MUTED     = '#64748B'
COLOR_EPAS1     = '#7C3AED'   # 紫色专属 EPAS1


def prettify_axis(ax, grid_axis='y'):
    ax.set_facecolor('white')
    for side in ('top', 'right'):
        ax.spines[side].set_visible(False)
    for side in ('left', 'bottom'):
        ax.spines[side].set_color('#94A3B8')
        ax.spines[side].set_linewidth(1.0)
    if grid_axis:
        ax.grid(axis=grid_axis, linestyle='--', linewidth=0.6,
                color='#E5E7EB', alpha=0.7, zorder=0)
    ax.tick_params(colors=COLOR_TEXT, length=5, width=1.0)


def add_subtitle(ax, text, y=1.02):
    ax.text(0, y, text, transform=ax.transAxes,
            ha='left', va='bottom', fontsize=17.3,
            color=COLOR_MUTED, style='italic')


def bh_fdr(pvals):
    pvals = np.asarray(pvals, dtype=float)
    if pvals.size == 0:
        return pvals
    order = np.argsort(pvals)
    ranked = pvals[order]
    adj = ranked * len(ranked) / (np.arange(len(ranked)) + 1)
    adj = np.minimum.accumulate(adj[::-1])[::-1]
    adj = np.clip(adj, 0, 1)
    out = np.empty_like(adj)
    out[order] = adj
    return out


def rankbiserial(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    x = x[np.isfinite(x)]
    y = y[np.isfinite(y)]
    if len(x) == 0 or len(y) == 0:
        return np.nan
    u_stat, _ = mannwhitneyu(x, y, alternative='two-sided')
    return 1 - 2 * u_stat / (len(x) * len(y))


def majority_vote(values):
    vals = pd.Series(values).dropna().astype(str)
    if vals.empty:
        return np.nan
    return vals.value_counts().index[0]


def direction_text(delta):
    if delta > 0:
        return 'Deviating higher'
    if delta < 0:
        return 'Normal higher'
    return 'No shift'


print('环境初始化完成')

## 1. 参数与路径配置

In [ ]:
# ===== 路径 =====
RNA_H5AD = Path(
    '/data/beifen/zhongmin/slide-tag/H5AD格式数据/'
    'slide-tag肺_with_scanvi_加上补测数据_对3级淋巴结构进行分类_186_2025_12_30.h5ad'
)
METAB_H5AD = Path(
    '/data/beifen/zhongmin/slide-tag/代谢/scCellFie/2026_3_24/'
    'slide-tag_lung_scCellFie_metabolic_tasks.h5ad'
)
RESULTS_DIR = Path(
    '/data/beifen/zhongmin/slide-tag/代谢/scCellFie/优化比较代谢差异/pyscenic'
)
TASK_BY_GENE_FNAME = Path(
    '/data/beifen/zhongmin/ref/scCellFie/task_data/homo_sapiens/Task_by_Gene.csv'
)

# pySCENIC 已有结果文件（复用）
ADJ_FNAME     = RESULTS_DIR / 'tls_corePlusAround_NvsD.adjacencies.tsv'
MOTIFS_FNAME  = RESULTS_DIR / 'tls_corePlusAround_NvsD.motifs.csv'
AUC_CSV       = RESULTS_DIR / 'Normal_vs_Deviating_corePlusAround_auc_matrix.csv'
REGULON_DIFF_CSV = RESULTS_DIR / 'Normal_vs_Deviating_corePlusAround_regulon_diff.csv'

# 代谢模块差异 / 代谢模块基因差异（按最终 4 模块 / 13 条通路重算并保存）
MODULE_DIFF_CSV       = RESULTS_DIR / 'Normal_vs_Deviating_corePlusAround_metabolic_module_diff.csv'
MODULE_GENE_DIFF_CSV  = RESULTS_DIR / 'Normal_vs_Deviating_corePlusAround_module_gene_diff.csv'
TLS_METAB_MODULE_CSV  = RESULTS_DIR / 'Normal_vs_Deviating_corePlusAround_tls_metabolic_module_scores.csv'
TLS_METAB_TASK_CSV    = RESULTS_DIR / 'Normal_vs_Deviating_corePlusAround_tls_metabolic_task_scores.csv'
TLS_AUC_MEDIAN_CSV    = RESULTS_DIR / 'Normal_vs_Deviating_corePlusAround_tls_median_auc.csv'

# 输出文件
EPAS1_OUT_DIR = RESULTS_DIR / 'EPAS1_rescue'
EPAS1_OUT_DIR.mkdir(parents=True, exist_ok=True)

EPAS1_REGULON_CSV       = EPAS1_OUT_DIR / 'EPAS1_manual_regulon_targets.csv'
EPAS1_TLS_SCORE_CSV     = EPAS1_OUT_DIR / 'EPAS1_tls_scores.csv'
EPAS1_METAB_OVERLAP_CSV = EPAS1_OUT_DIR / 'EPAS1_metabolic_module_overlap.csv'
EPAS1_CORR_CSV          = EPAS1_OUT_DIR / 'EPAS1_module_correlation.csv'
EPAS1_GENE_DIFF_CSV     = EPAS1_OUT_DIR / 'EPAS1_metabolic_target_gene_diff.csv'

# 图片输出
FIG_VOLCANO_PDF  = EPAS1_OUT_DIR / 'EPAS1_volcano_in_context.pdf'
FIG_EXPR_PDF     = EPAS1_OUT_DIR / 'EPAS1_expression_Normal_vs_Deviating.pdf'
FIG_SCORE_PDF    = EPAS1_OUT_DIR / 'EPAS1_regulon_score_Normal_vs_Deviating.pdf'
FIG_CORR_PDF     = EPAS1_OUT_DIR / 'EPAS1_metabolic_module_correlation.pdf'
FIG_NETWORK_PDF  = EPAS1_OUT_DIR / 'EPAS1_metabolic_gene_network.pdf'
FIG_HEATMAP_PDF  = EPAS1_OUT_DIR / 'EPAS1_target_gene_heatmap.pdf'

# ===== 分析参数 =====
TLS_ID_COL    = 'tls_degrow_id_sample'
TLS_CLASS_COL = 'tls_degrow_id_fenlei'
AROUND_COL    = 'tls_degrow_id_around'
SAMPLE_COL    = 'sample'

NORMAL_CLASSES   = {'Conforming', 'Mature'}
DEVIATING_CLASS  = 'Deviating'
SCENIC_SCOPE     = 'core_plus_around'

# EPAS1 手动 regulon 参数
# 只用 importance 最高的前 N 个靶基因构建 pseudo-regulon（去掉 ENSEMBL ID，只保留 HGNC 符号）
EPAS1_TOP_N_TARGETS   = 200    # 手动 regulon 的靶基因数（取 importance 前 200）
EPAS1_MIN_IMPORTANCE  = 0.1    # importance 阈值（同时满足 top-N 和此阈值）
MIN_GENE_CELLS        = 10

# 代谢模块定义（最终版本：4 个模块 / 13 条代谢通路）
MODULES = {
    'M1: PIP2-IP3 Ca2+': [
        'Conversion of 1-phosphatidyl-1D-myo-inositol 4,5-bisphosphate to 1D-myo-inositol 1,4,5-trisphosphate',
        'Phosphatidyl-inositol synthesis',
    ],
    'M2: Glycan maturation': [
        'Sialylation (addition of sialic acid)',
        'Branching (N-acetylglucosaminyltransferases)',
        'Fucosylation (addition of fucose)',
    ],
    'M3: Arg-Gln axis': [
        'Arginine synthesis',
        'Conversion of aspartate to arginine',
        'Arginine degradation',
    ],
    'M4: Lipid-Cardiolipin': [
        'Cardiolipin synthesis',
        'Triacylglycerol synthesis',
        'Synthesis of palmitoyl-CoA',
        'Phosphatidyl-serine synthesis',
        'Synthesis of glucocerebroside',
    ],
}
MODULE_COLORS = {
    'M1: PIP2-IP3 Ca2+':      '#E41A1C',
    'M2: Glycan maturation':  '#377EB8',
    'M3: Arg-Gln axis':       '#FF7F00',
    'M4: Lipid-Cardiolipin':  '#984EA3',
}
MODULE_ORDER = list(MODULES.keys())
FINAL_TASKS = [task for module in MODULE_ORDER for task in MODULES[module]]
TASK_TO_MODULE = {
    task: module
    for module, tasks in MODULES.items()
    for task in tasks
}

print('参数配置完成')
print(f'EPAS1 手动 regulon: top {EPAS1_TOP_N_TARGETS} 靶基因 | importance ≥ {EPAS1_MIN_IMPORTANCE}')

## 2. 加载数据

In [ ]:
# ===== 辅助函数：构建 TLS 子集 =====
def _norm_class(x):
    if x is None:
        return '<NA>'
    s = str(x).strip()
    if s == '' or s.lower() in {'nan', '<na>', 'none'}:
        return '<NA>'
    low = s.lower()
    if 'conform' in low: return 'Conforming'
    if 'mature'  in low: return 'Mature'
    if 'deviat'  in low: return 'Deviating'
    return s


def _is_valid_id(x):
    s = str(x).strip()
    return s not in {'', 'nan', '<NA>', 'None', 'NA', 'N/A', 'none'}


def _around_to_core_id(x):
    if not _is_valid_id(x):
        return None
    s = re.sub(r'_around.*$', '', str(x).strip())
    return s if _is_valid_id(s) else None


def build_tls_subset(obs, scope='core_plus_around'):
    cols = [TLS_ID_COL, TLS_CLASS_COL, SAMPLE_COL]
    if AROUND_COL in obs.columns:
        cols.append(AROUND_COL)
    out = obs[cols].copy()
    for c in cols:
        out[c] = out[c].astype(str)
    if AROUND_COL not in out.columns:
        out[AROUND_COL] = ''

    out['tls_class_norm'] = out[TLS_CLASS_COL].map(_norm_class)
    out['around_core_id'] = out[AROUND_COL].map(_around_to_core_id)

    core_mask = (out[TLS_ID_COL].map(_is_valid_id)
                 & out['tls_class_norm'].isin(NORMAL_CLASSES | {DEVIATING_CLASS}))
    core_df   = out.loc[core_mask, [TLS_ID_COL, 'tls_class_norm']].copy()
    tls2class = (core_df.groupby(TLS_ID_COL, observed=True)['tls_class_norm']
                 .agg(majority_vote).dropna().to_dict())
    selected_tls = set(tls2class)

    if scope == 'core_plus_around':
        keep = out[TLS_ID_COL].isin(selected_tls) | out['around_core_id'].isin(selected_tls)
        out  = out.loc[keep].copy()
        out['tls_link'] = np.where(
            out[TLS_ID_COL].isin(selected_tls),
            out[TLS_ID_COL].to_numpy(),
            out['around_core_id'].to_numpy(),
        )
        out['tls_scope'] = np.where(out[TLS_ID_COL].isin(selected_tls), 'core', 'around')
    else:
        out = out.loc[out[TLS_ID_COL].isin(selected_tls)].copy()
        out['tls_link']  = out[TLS_ID_COL].astype(str)
        out['tls_scope'] = 'core'

    out['tls_group'] = out['tls_link'].map(
        lambda x: 'Normal' if tls2class.get(x) in NORMAL_CLASSES
                  else ('Deviating' if tls2class.get(x) == DEVIATING_CLASS else '<NA>')
    )
    out = out.loc[
        out['tls_group'].isin(['Normal', 'Deviating'])
        & out['tls_link'].map(_is_valid_id)
    ].copy()
    return out, tls2class


# ===== 加载 RNA =====
print('加载 RNA h5ad ...')
adata = sc.read_h5ad(str(RNA_H5AD))
print(f'  RNA: {adata.n_obs} cells × {adata.n_vars} genes')

# ===== 加载代谢 =====
print('加载代谢 h5ad ...')
metab_adata = sc.read_h5ad(str(METAB_H5AD))
print(f'  Metab: {metab_adata.n_obs} cells × {metab_adata.n_vars} tasks')

# ===== 加载 Task_by_Gene =====
task_by_gene = pd.read_csv(str(TASK_BY_GENE_FNAME), index_col=0)
print(f'  Task_by_Gene: {task_by_gene.shape[0]} tasks × {task_by_gene.shape[1]} genes')

# ===== 构建 TLS 子集 =====
tls_obs, tls2class = build_tls_subset(adata.obs, scope=SCENIC_SCOPE)
tls_adata = adata[tls_obs.index].copy()
tls_adata.obs = tls_obs.loc[tls_adata.obs_names].copy()
tls_adata.var_names_make_unique()
sc.pp.filter_genes(tls_adata, min_cells=MIN_GENE_CELLS)

print(f'\nTLS subset: {tls_adata.n_obs} cells, {tls_adata.n_vars} genes')
print(tls_adata.obs['tls_group'].value_counts().to_string())
print(f'EPAS1 in TLS subset: {"EPAS1" in tls_adata.var_names}')

## 4. 构建 EPAS1 手动 Regulon 并计算 TLS 水平活性分

In [ ]:
# ===== 4.1 筛选 EPAS1 高可信靶基因 =====
# 排除 ENSEMBL ID（非 HGNC 符号）
epas1_adj_hgnc = epas1_adj[
    ~epas1_adj['target'].str.startswith('ENSG')
    & (epas1_adj['importance'] >= EPAS1_MIN_IMPORTANCE)
].copy()

# 只保留在 TLS RNA 数据中实际存在的基因
epas1_adj_hgnc = epas1_adj_hgnc[
    epas1_adj_hgnc['target'].isin(tls_adata.var_names)
].copy().reset_index(drop=True)

epas1_targets_all = epas1_adj_hgnc.head(EPAS1_TOP_N_TARGETS)['target'].tolist()
print(f'EPAS1 手动 regulon 靶基因数: {len(epas1_targets_all)}')
print(f'  (过滤后可用: {len(epas1_adj_hgnc)} | 选取 top {EPAS1_TOP_N_TARGETS})')
print(f'  Top 20 靶基因: {epas1_targets_all[:20]}')

epas1_adj_hgnc.head(EPAS1_TOP_N_TARGETS).to_csv(str(EPAS1_REGULON_CSV), index=False)
print(f'[Saved] {EPAS1_REGULON_CSV}')


# ===== 4.2 计算 TLS-level EPAS1 分数 =====
# 方法1：直接用 EPAS1 RNA 表达（log1p count）
# 方法2：手动 regulon 靶基因均值表达
# 方法3：AUCell 风格（rank-based AUC）

def compute_aucell_score(X_log, target_gene_idx, auc_threshold_frac=0.05):
    """
    对每个细胞计算 AUCell 风格分：
    - 对每个细胞的所有基因按表达量降序排名
    - 计算前 k 个基因 (k = auc_threshold_frac * n_genes) 中
      包含靶基因的比例 AUC
    """
    n_cells, n_genes = X_log.shape
    k = max(1, int(np.ceil(auc_threshold_frac * n_genes)))
    n_targets = len(target_gene_idx)
    scores = np.zeros(n_cells, dtype=np.float32)
    for i in range(n_cells):
        row = X_log[i, :]
        top_idx = np.argpartition(row, -k)[-k:]
        hit = np.isin(top_idx, target_gene_idx).sum()
        # AUC = 命中数 / 靶基因总数（规范化）
        scores[i] = hit / n_targets if n_targets > 0 else 0.0
    return scores


print('\n计算 TLS 细胞水平的 EPAS1 分数（可能需要 1-2 分钟）...')
tls_gene_idx = {g: i for i, g in enumerate(tls_adata.var_names)}

# 获取表达矩阵
X_tls = tls_adata.X
if issparse(X_tls):
    X_tls = X_tls.toarray()
X_tls = np.asarray(X_tls, dtype=np.float32)

# Method 1: EPAS1 自身表达
if 'EPAS1' in tls_gene_idx:
    epas1_expr_cells = X_tls[:, tls_gene_idx['EPAS1']]
else:
    epas1_expr_cells = np.zeros(tls_adata.n_obs, dtype=np.float32)
    print('[Warn] EPAS1 在 TLS subset 中未检测到（可能被 filter_genes 过滤）')

# Method 2: 靶基因均值表达分
target_idx_list = [tls_gene_idx[g] for g in epas1_targets_all if g in tls_gene_idx]
print(f'  靶基因中在矩阵里找到: {len(target_idx_list)} / {len(epas1_targets_all)}')
regulon_mean_cells = X_tls[:, target_idx_list].mean(axis=1)

# Method 3: AUCell 风格分
print('  计算 AUCell 风格分...')
aucell_scores_cells = compute_aucell_score(X_tls, target_idx_list, auc_threshold_frac=0.05)

# 汇总到 cell DataFrame
cell_score_df = pd.DataFrame({
    'EPAS1_expr':         epas1_expr_cells,
    'EPAS1_target_mean':  regulon_mean_cells,
    'EPAS1_aucell':       aucell_scores_cells,
    'tls_link':   tls_adata.obs['tls_link'].values,
    'tls_group':  tls_adata.obs['tls_group'].values,
    'tls_scope':  tls_adata.obs['tls_scope'].values,
    'sample':     tls_adata.obs[SAMPLE_COL].values,
}, index=tls_adata.obs_names)

# 聚合到 TLS 水平（中位数）
score_cols = ['EPAS1_expr', 'EPAS1_target_mean', 'EPAS1_aucell']
tls_epas1_scores = cell_score_df.groupby('tls_link', observed=True)[score_cols].median()
tls_epas1_group = cell_score_df.groupby('tls_link', observed=True)['tls_group'].agg(majority_vote)
tls_epas1_scores['tls_group'] = tls_epas1_group

# 过滤有效 TLS（组别明确）
tls_epas1_scores = tls_epas1_scores[
    tls_epas1_scores['tls_group'].isin(['Normal', 'Deviating'])
].copy()

tls_epas1_scores.to_csv(str(EPAS1_TLS_SCORE_CSV))
print(f'\nTLS-level EPAS1 分数计算完成')
print(f'  Normal TLS: {(tls_epas1_scores["tls_group"]=="Normal").sum()}')
print(f'  Deviating TLS: {(tls_epas1_scores["tls_group"]=="Deviating").sum()}')
print(f'[Saved] {EPAS1_TLS_SCORE_CSV}')

## 5. EPAS1 在 Normal vs Deviating TLS 中的差异检验

In [ ]:
epas1_diff_results = {}

for score_name, score_label in [
    ('EPAS1_expr',        'EPAS1 RNA expression (median per TLS)'),
    ('EPAS1_target_mean', 'EPAS1 target gene mean expression'),
    ('EPAS1_aucell',      'EPAS1 AUCell-style regulon score'),
]:
    x_n = tls_epas1_scores.loc[tls_epas1_scores['tls_group'] == 'Normal',  score_name].to_numpy(dtype=float)
    x_d = tls_epas1_scores.loc[tls_epas1_scores['tls_group'] == 'Deviating', score_name].to_numpy(dtype=float)
    x_n = x_n[np.isfinite(x_n)]
    x_d = x_d[np.isfinite(x_d)]
    _, pval = mannwhitneyu(x_n, x_d, alternative='two-sided')
    delta = float(np.median(x_d) - np.median(x_n))
    rbc   = rankbiserial(x_n, x_d)
    epas1_diff_results[score_name] = {
        'label':            score_label,
        'median_Normal':    float(np.median(x_n)),
        'median_Deviating': float(np.median(x_d)),
        'delta':            delta,
        'rank_biserial':    rbc,
        'p_value':          pval,
        'direction':        direction_text(delta),
        'n_Normal':         len(x_n),
        'n_Deviating':      len(x_d),
    }

epas1_diff_df = pd.DataFrame(epas1_diff_results).T
print('[EPAS1 Normal vs Deviating TLS 差异检验]')
print(epas1_diff_df[['label','median_Normal','median_Deviating','delta',
                       'rank_biserial','p_value','direction']].to_string())

## 6. 图1：EPAS1 三种活性度量 — Normal vs Deviating TLS 比较

> **为什么同一组比较有3个 Y 轴不同的小图？**
>
> 因为 EPAS1 被 pySCENIC CTX 步骤过滤，无法直接用 AUC 表示其活性，所以这里从 **3 个不同角度** 衡量 EPAS1 在每个 TLS 中的活性，3 种度量单位不同，Y 轴自然不同：
>
> | 面板 | 度量内容 | Y 轴含义 |
> |------|---------|----------|
> | **左图** | **EPAS1 基因自身 RNA 表达**（每个细胞的 log-count，取 TLS 内中位数） | log-count（越高 = EPAS1 mRNA 越多） |
> | **中图** | **EPAS1 GRN 靶基因（top-200）的平均表达量**（反映靶基因整体共激活水平） | 靶基因均值 log-count |
> | **右图** | **AUCell 风格 regulon 活性分**：每个细胞中，EPAS1 top-200 靶基因落在全基因组表达排名前 5% 的比例（与 pySCENIC AUCell 输出格式一致） | AUC 分（0~1，越高 = EPAS1 靶基因整体越活跃） |
>
> **三张图结论一致**（均显示 Deviating TLS 中 EPAS1 活性更高），从不同角度互相印证，比单一指标更可信。  
> 后续分析以右图的 **AUCell 分**作为 EPAS1 的代表性活性指标（与 pySCENIC 输出格式一致）。

In [ ]:
PLOT_NORMAL = '#6E8DD8'
PLOT_DEVIATING = '#D97A73'
PLOT_EPAS1 = '#6D28D9'
PLOT_GRID = '#E2E8F0'

fig, axes = plt.subplots(1, 3, figsize=(14.6, 5.8))
plt.subplots_adjust(left=0.07, right=0.985, top=0.78, bottom=0.19, wspace=0.34)

score_configs = [
    ('EPAS1_expr',        'EPAS1 mRNA',              'median log-count / TLS', 'EPAS1_expr',        'I'),
    ('EPAS1_target_mean', 'GRN target mean',         'mean expression / TLS',  'EPAS1_target_mean', 'II'),
    ('EPAS1_aucell',      'Regulon AUCell',          'rank-based score (0–1)', 'EPAS1_aucell',      'III'),
]

for ax, (col, title_txt, ylabel_sub, key, panel_num) in zip(axes, score_configs):
    prettify_axis(ax, grid_axis='y')
    ax.grid(axis='y', linestyle=(0, (2.2, 2.2)), linewidth=0.7,
            color=PLOT_GRID, alpha=0.9, zorder=0)

    plot_data = tls_epas1_scores[tls_epas1_scores['tls_group'].isin(['Normal', 'Deviating'])].copy()
    group_order = ['Normal', 'Deviating']
    colors = [PLOT_NORMAL, PLOT_DEVIATING]

    vals_by_group = [
        plot_data.loc[plot_data['tls_group'] == g, col].to_numpy(dtype=float)
        for g in group_order
    ]

    parts = ax.violinplot(vals_by_group, positions=[0, 1],
                          showmedians=True, showextrema=False, widths=0.58)
    for pc, color in zip(parts['bodies'], colors):
        pc.set_facecolor(mpl.colors.to_rgba(color, 0.34))
        pc.set_edgecolor(mpl.colors.to_rgba(color, 0.85))
        pc.set_linewidth(1.1)
    parts['cmedians'].set_color(COLOR_TEXT)
    parts['cmedians'].set_linewidth(2.2)
    parts['cmedians'].set_zorder(6)

    rng = np.random.default_rng(2026)
    for xi, (g, color) in enumerate(zip(group_order, colors)):
        vals = plot_data.loc[plot_data['tls_group'] == g, col].to_numpy(dtype=float)
        jitter = rng.uniform(-0.085, 0.085, len(vals))
        ax.scatter(xi + jitter, vals, s=28,
                   color=mpl.colors.to_rgba(color, 0.72),
                   edgecolors='white', linewidths=0.55, zorder=4)

    for xi, (vals, color) in enumerate(zip(vals_by_group, colors)):
        q1, q3 = np.percentile(vals, [25, 75])
        med = np.median(vals)
        ax.vlines(xi, q1, q3, lw=5.2, color='white', zorder=5)
        ax.vlines(xi, q1, q3, lw=3.2, color=color, alpha=0.85, zorder=5)
        ax.hlines(med, xi - 0.11, xi + 0.11, color=COLOR_TEXT, lw=1.8, zorder=6)

    res = epas1_diff_results[key]
    pval = res['p_value']
    star = '****' if pval < 1e-4 else ('***' if pval < 0.001 else ('**' if pval < 0.01 else ('*' if pval < 0.05 else 'ns')))
    y_max = max(np.max(vals_by_group[0]), np.max(vals_by_group[1]))
    y_min = min(np.min(vals_by_group[0]), np.min(vals_by_group[1]))
    y_span = max(y_max - y_min, max(abs(y_max), 1.0) * 0.20)
    y_line = y_max + y_span * 0.16
    ax.plot([0, 1], [y_line, y_line], color=COLOR_TEXT, lw=1.25)
    ax.plot([0, 0], [y_max + y_span * 0.03, y_line], color=COLOR_TEXT, lw=1.05)
    ax.plot([1, 1], [y_max + y_span * 0.03, y_line], color=COLOR_TEXT, lw=1.05)
    ax.text(0.5, y_line + y_span * 0.018, f'{star}  p={pval:.2e}',
            ha='center', va='bottom', fontsize=19.3,
            fontweight='bold', color=COLOR_TEXT)

    ax.set_xticks([0, 1])
    ax.set_xticklabels(['Normal', 'Deviating'], fontsize=22.7, fontweight='bold')
    ax.tick_params(axis='x', length=0)
    ax.set_ylabel(f'{title_txt}\n({ylabel_sub})', fontsize=20.7, labelpad=8)
    ax.set_xlim(-0.55, 1.55)
    ax.set_ylim(y_min - y_span * 0.08, y_max + y_span * 0.34)
    ax.yaxis.set_major_locator(mpl.ticker.MaxNLocator(5))

    ax.text(-0.18, 1.08, panel_num, transform=ax.transAxes,
            fontsize=28, fontweight='bold', color=COLOR_TEXT, va='top')
    ax.set_title(title_txt, fontsize=22.4, fontweight='bold',
                 color=COLOR_TEXT, pad=12)
    ax.text(0.00, 1.025, f'N={res["n_Normal"]} Normal   |   D={res["n_Deviating"]} Deviating',
            transform=ax.transAxes, ha='left', va='bottom', fontsize=15.7,
            color=COLOR_MUTED, style='italic')
    ax.text(1.00, 1.025, f'RBC={res["rank_biserial"]:+.2f}',
            transform=ax.transAxes, ha='right', va='bottom', fontsize=15.7,
            color=COLOR_MUTED, style='italic')

fig.suptitle('EPAS1 Readouts Across Normal and Deviating TLS',
             fontsize=26.7, fontweight='bold', color=COLOR_TEXT, y=0.975)
fig.text(0.5, 0.90,
         'RNA abundance, direct-target expression, and AUCell-style regulon activity shown side-by-side',
         ha='center', va='center', fontsize=16.8, color=COLOR_MUTED, style='italic')
fig.text(0.5, 0.075,
         'Panel I, TLS-median EPAS1 RNA   |   Panel II, mean expression of top-200 EPAS1 GRN targets   |   Panel III, AUCell-style regulon score',
         ha='center', va='center', fontsize=15.6, color=COLOR_MUTED, style='italic')

plt.savefig(str(FIG_SCORE_PDF), dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print(f'[Saved] {FIG_SCORE_PDF}')


## 7. 图2：将 EPAS1 标注到原有 regulon 火山图中

虽然 EPAS1 不在 pySCENIC 的 AUCell 结果中，我们把它用特殊颜色加入火山图，
使用我们手动构建的分数作为替代。

In [ ]:
"#e41a1c",
"#377eb8",
"#ff7f00",
"#984ea3",

In [ ]:
# 读取已有 regulon 差异结果
diff_df = pd.read_csv(str(REGULON_DIFF_CSV))

VOLC_DEVIATING = '#e41a1c'
VOLC_NORMAL = '#377eb8'
VOLC_NEUTRAL = '#ff7f00'
VOLC_EPAS1 = '#984ea3'

def clean_tf_name(regulon_name):
    name = str(regulon_name)
    name = re.sub(r'[_ ]?\(\+\)$', '', name)
    name = re.sub(r'[_ ]?\(-\)$', '', name)
    return name

diff_df['tf_name'] = diff_df['regulon'].map(clean_tf_name)
diff_df['neglog10_fdr'] = -np.log10(diff_df['fdr'].clip(lower=1e-300))
diff_df['is_sig'] = diff_df['fdr'] < 0.05

epas1_manual_res = epas1_diff_results['EPAS1_aucell']
epas1_manual_p = epas1_manual_res['p_value']
epas1_manual_neg = -np.log10(max(epas1_manual_p, 1e-300))
epas1_manual_d = epas1_manual_res['delta']

fig, ax = plt.subplots(figsize=(9.6, 7.1))
prettify_axis(ax, grid_axis=None)
for side in ('left', 'bottom'):
    ax.spines[side].set_linewidth(0.55)

bg = diff_df[~diff_df['is_sig']]
ax.scatter(bg['delta_median(Dev-Norm)'], bg['neglog10_fdr'],
           s=170, color=VOLC_NEUTRAL, alpha=0.55, linewidths=0, zorder=1)

sig_d = diff_df[diff_df['is_sig'] & (diff_df['delta_median(Dev-Norm)'] > 0)]
sig_n = diff_df[diff_df['is_sig'] & (diff_df['delta_median(Dev-Norm)'] <= 0)]
ax.scatter(sig_d['delta_median(Dev-Norm)'], sig_d['neglog10_fdr'],
           s=270, color=VOLC_DEVIATING, alpha=0.95,
           edgecolors='white', linewidths=0.45, zorder=3,
           label='Deviating higher')
ax.scatter(sig_n['delta_median(Dev-Norm)'], sig_n['neglog10_fdr'],
           s=270, color=VOLC_NORMAL, alpha=0.95,
           edgecolors='white', linewidths=0.45, zorder=3,
           label='Normal higher')

fdr_line = -np.log10(0.05)
ax.axvline(0, color='#6B7280', lw=0.55, ls=(0, (3.2, 3.2)), zorder=0)
ax.axhline(fdr_line, color='#6B7280', lw=0.55, ls=(0, (1.5, 2.4)), zorder=0)

x_min = float(diff_df['delta_median(Dev-Norm)'].min())
x_max = float(diff_df['delta_median(Dev-Norm)'].max())
y_max = float(diff_df['neglog10_fdr'].max())
x_span = max(x_max - x_min, 0.10)
y_span = max(y_max, 1.0)
ax.set_xlim(x_min - 0.04 * x_span, x_max + 0.12 * x_span)
ax.set_ylim(-0.05, y_max + 0.55)
ax.xaxis.set_major_locator(mpl.ticker.MultipleLocator(0.05))
ax.yaxis.set_major_locator(mpl.ticker.MultipleLocator(2))
ax.xaxis.set_minor_locator(mpl.ticker.NullLocator())
ax.yaxis.set_minor_locator(mpl.ticker.NullLocator())
ax.tick_params(axis='both', which='major', labelsize=20, width=0.55, length=4)
ax.text(ax.get_xlim()[1] - 0.02 * x_span, fdr_line + 0.08, 'FDR = 0.05',
        ha='right', va='bottom', fontsize=17.3, color='#94A3B8')

def _annotate_ranked(df_side, side):
    if df_side.empty:
        return
    df_side = df_side.sort_values('neglog10_fdr', ascending=False).reset_index(drop=True)
    n = len(df_side)
    y_top = ax.get_ylim()[1] - 0.18
    y_bot = max(fdr_line + 1.55, y_top - 0.46 * max(n - 1, 0))
    y_targets = np.linspace(y_top, y_bot, n)

    for i, (_, row) in enumerate(df_side.iterrows()):
        color = VOLC_DEVIATING if row['delta_median(Dev-Norm)'] > 0 else VOLC_NORMAL
        if side == 'right':
            x_text = row['delta_median(Dev-Norm)'] + (0.018 + 0.008 * i) * x_span
            ha = 'left'
        else:
            x_text = row['delta_median(Dev-Norm)'] - (0.018 + 0.008 * i) * x_span
            ha = 'right'

        ax.annotate(
            row['tf_name'],
            xy=(row['delta_median(Dev-Norm)'], row['neglog10_fdr']),
            xytext=(x_text, y_targets[i]),
            fontsize=18.4, fontweight='bold', color=color,
            ha=ha, va='center',
            arrowprops=dict(arrowstyle='-', color='#A0AEC0', lw=0.45),
            bbox=dict(boxstyle='round,pad=0.24', fc='white',
                      ec=mpl.colors.to_rgba(color, 0.35), lw=0.35, alpha=0.98),
            zorder=8,
        )

top_pos = sig_d.nlargest(3, 'neglog10_fdr')
top_neg = sig_n.nlargest(3, 'neglog10_fdr')
_annotate_ranked(top_neg, 'left')
_annotate_ranked(top_pos, 'right')

ax.scatter([epas1_manual_d], [epas1_manual_neg],
           s=780, color=mpl.colors.to_rgba(VOLC_EPAS1, 0.12),
           marker='o', linewidths=0, zorder=9)
ax.scatter([epas1_manual_d], [epas1_manual_neg],
           s=460, color=VOLC_EPAS1, marker='*', zorder=10,
           edgecolors='white', linewidths=0.55,
           label='EPAS1')
ax.annotate(
    'EPAS1',
    xy=(epas1_manual_d, epas1_manual_neg),
    xytext=(epas1_manual_d + 0.030 * x_span, epas1_manual_neg + 0.16 * y_span),
    fontsize=20.7, fontweight='bold', color=VOLC_EPAS1,
    ha='left', va='center',
    arrowprops=dict(arrowstyle='->', color=VOLC_EPAS1, lw=0.75),
    bbox=dict(boxstyle='round,pad=0.36',
              fc=mpl.colors.to_rgba(VOLC_EPAS1, 0.07),
              ec=mpl.colors.to_rgba(VOLC_EPAS1, 0.78), lw=0.45),
    zorder=12,
)

ax.set_xlabel('Delta median AUC (Deviating − Normal)', fontsize=21.6)
ax.set_ylabel('−log₁₀(FDR)', fontsize=21.6)
ax.set_title('Regulon Activity Shift Between Normal and Deviating TLS',
             fontsize=24.7, fontweight='bold', pad=14, color=COLOR_TEXT)
add_subtitle(ax, 'pySCENIC CTX-pruned regulons with EPAS1 highlighted', y=1.01)
ax.legend(loc='upper left', fontsize=17.3, frameon=False, handletextpad=0.7)

plt.tight_layout()
plt.savefig(str(FIG_VOLCANO_PDF), dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print(f'[Saved] {FIG_VOLCANO_PDF}')


## 8. EPAS1 靶基因 × 代谢模块基因的重叠

In [ ]:
# ===== 重要说明 =====
# 这里不再复用旧的 5 模块 / 25 通路结果，而是按最终确定的 4 模块 / 13 条通路重算。
# 直接从 TLS-level metabolic task scores 中抽取最终通路，并重建模块分数、模块差异、模块基因差异。
# ─────────────────────────────────────────────────────────────────────────────
EPAS1_MIN_IMP_NETWORK = 0.05   # 网络分析专用：更低门槛，覆盖更多代谢基因

epas1_adj_network = epas1_adj[
    ~epas1_adj['target'].str.startswith('ENSG')
    & (epas1_adj['importance'] >= EPAS1_MIN_IMP_NETWORK)
    & epas1_adj['target'].isin(tls_adata.var_names)
].copy().sort_values('importance', ascending=False).reset_index(drop=True)

epas1_network_targets = set(epas1_adj_network['target'].tolist())
print(f'网络分析用靶基因数 (importance≥{EPAS1_MIN_IMP_NETWORK}): {len(epas1_network_targets)}')
print(f'AUCell 打分用靶基因数 (top-200, importance≥0.629): {len(epas1_targets_all)}')

# ── 1) 读取 TLS-level task scores，并只保留最终 13 条通路 ─────────────────────
tls_task_df = pd.read_csv(str(TLS_METAB_TASK_CSV), index_col=0)
missing_tasks = [t for t in FINAL_TASKS if t not in tls_task_df.columns]
if missing_tasks:
    raise KeyError(f'以下最终通路不在 TLS task score 文件中: {missing_tasks}')

common_tls_task = tls_epas1_scores.index.intersection(tls_task_df.index)
common_tls_task = [
    t for t in common_tls_task
    if tls_epas1_scores.loc[t, 'tls_group'] in ['Normal', 'Deviating']
]
tls_task_df = tls_task_df.loc[common_tls_task, FINAL_TASKS].copy()
tls_task_df.to_csv(str(TLS_METAB_TASK_CSV))
print()
print(f'TLS-level final task scores: {tls_task_df.shape[0]} TLS × {tls_task_df.shape[1]} tasks')

# ── 2) 重建 TLS-level metabolic module scores ─────────────────────────────────
module_score_df = pd.DataFrame(index=tls_task_df.index)
for module in MODULE_ORDER:
    tasks_present = [t for t in MODULES[module] if t in tls_task_df.columns]
    if not tasks_present:
        continue
    module_score_df[module] = tls_task_df[tasks_present].mean(axis=1)
module_score_df['tls_group'] = tls_epas1_scores.loc[module_score_df.index, 'tls_group'].astype(str).values
module_score_df.to_csv(str(TLS_METAB_MODULE_CSV))

# ── 3) 重算 module-level Normal vs Deviating 差异 ─────────────────────────────
module_rows = []
for module in MODULE_ORDER:
    if module not in module_score_df.columns:
        continue
    vals_n = module_score_df.loc[module_score_df['tls_group']=='Normal', module].dropna().to_numpy(dtype=float)
    vals_d = module_score_df.loc[module_score_df['tls_group']=='Deviating', module].dropna().to_numpy(dtype=float)
    if len(vals_n) == 0 or len(vals_d) == 0:
        continue
    u_stat, pval = mannwhitneyu(vals_d, vals_n, alternative='two-sided')
    rbc = (2 * u_stat / (len(vals_d) * len(vals_n))) - 1
    med_n = float(np.median(vals_n))
    med_d = float(np.median(vals_d))
    delta = med_d - med_n
    module_rows.append({
        'module': module,
        'n_tasks_used': len([t for t in MODULES[module] if t in tls_task_df.columns]),
        'median_Normal': med_n,
        'median_Deviating': med_d,
        'delta_median(Dev-Norm)': delta,
        'rank_biserial': float(rbc),
        'p_value': float(pval),
        'direction': direction_text(delta),
    })
module_diff = pd.DataFrame(module_rows)
module_diff['fdr'] = bh_fdr(module_diff['p_value'].to_numpy(dtype=float))
module_diff = module_diff.sort_values('module').reset_index(drop=True)
module_diff.to_csv(str(MODULE_DIFF_CSV), index=False)
print(f'[Saved] {MODULE_DIFF_CSV}')
print(module_diff[['module','n_tasks_used','delta_median(Dev-Norm)','p_value','fdr','direction']].to_string(index=False))

# ── 4) 重算 module gene differential expression（按 TLS-level gene mean 比较） ───────
# 这里不能直接用 cell-level 中位数差判断方向：很多稀疏基因两组细胞中位数都会是 0，
# 会把真实偏向压成 "No shift"，从而让网络图里几乎全变成灰色虚线。
# 因此改成：先在每个 TLS 内对基因表达取 mean，再比较 Normal vs Deviating TLS。
cell_groups = tls_adata.obs['tls_group'].astype(str).to_numpy()
cell_scopes = tls_adata.obs['tls_scope'].astype(str).to_numpy() if 'tls_scope' in tls_adata.obs.columns else np.repeat('core', tls_adata.n_obs)
idx_core = cell_scopes == 'core'
idx_around = cell_scopes == 'around'

tls_group_series = tls_adata.obs.groupby('tls_link', observed=True)['tls_group'].agg(majority_vote)
tls_group_series = tls_group_series[tls_group_series.isin(['Normal', 'Deviating'])]

module_gene_rows = []
for module in MODULE_ORDER:
    tasks_present = [t for t in MODULES[module] if t in task_by_gene.index]
    if not tasks_present:
        continue
    gene_mask = task_by_gene.loc[tasks_present].sum(axis=0) > 0
    module_genes = [g for g in task_by_gene.columns[gene_mask].astype(str).tolist() if g in tls_adata.var_names]
    if not module_genes:
        continue

    X_mod = tls_adata[:, module_genes].X
    if issparse(X_mod):
        X_mod = X_mod.toarray()
    X_mod = np.asarray(X_mod, dtype=np.float32)

    cell_expr_df = pd.DataFrame(X_mod, index=tls_adata.obs_names, columns=module_genes)
    cell_expr_df['tls_link'] = tls_adata.obs['tls_link'].values
    tls_gene_df = cell_expr_df.groupby('tls_link', observed=True)[module_genes].mean()
    tls_gene_df = tls_gene_df.loc[tls_gene_df.index.intersection(tls_group_series.index)].copy()
    tls_gene_group = tls_group_series.loc[tls_gene_df.index]

    for j, gene in enumerate(module_genes):
        tls_vals_n = tls_gene_df.loc[tls_gene_group == 'Normal', gene].to_numpy(dtype=float)
        tls_vals_d = tls_gene_df.loc[tls_gene_group == 'Deviating', gene].to_numpy(dtype=float)
        tls_vals_n = tls_vals_n[np.isfinite(tls_vals_n)]
        tls_vals_d = tls_vals_d[np.isfinite(tls_vals_d)]
        if len(tls_vals_n) == 0 or len(tls_vals_d) == 0:
            continue
        try:
            u_stat, pval = mannwhitneyu(tls_vals_d, tls_vals_n, alternative='two-sided')
            rbc = (2 * u_stat / (len(tls_vals_d) * len(tls_vals_n))) - 1
        except Exception:
            pval, rbc = 1.0, np.nan

        vec = X_mod[:, j].astype(float)
        med_n = float(np.median(tls_vals_n))
        med_d = float(np.median(tls_vals_d))
        delta = med_d - med_n
        module_gene_rows.append({
            'module': module,
            'gene': gene,
            'median_Normal': med_n,
            'median_Deviating': med_d,
            'delta_median(Dev-Norm)': delta,
            'rank_biserial': float(rbc) if np.isfinite(rbc) else np.nan,
            'p_value': float(pval),
            'cell_mean_all': float(np.nanmean(vec)),
            'det_frac_cells': float(np.mean(vec > 0)),
            'det_frac_core': float(np.mean(vec[idx_core] > 0)) if idx_core.any() else np.nan,
            'det_frac_around': float(np.mean(vec[idx_around] > 0)) if idx_around.any() else np.nan,
        })

module_gene_diff = pd.DataFrame(module_gene_rows)
module_gene_diff['fdr_global'] = bh_fdr(module_gene_diff['p_value'].to_numpy(dtype=float))
module_gene_diff['fdr_within_module'] = (
    module_gene_diff.groupby('module', group_keys=False)['p_value']
    .apply(lambda s: pd.Series(bh_fdr(s.to_numpy(dtype=float)), index=s.index))
)
module_gene_diff['direction'] = module_gene_diff['delta_median(Dev-Norm)'].map(direction_text)
module_gene_diff['abs_delta'] = module_gene_diff['delta_median(Dev-Norm)'].abs()
module_gene_diff['is_changed_gene'] = module_gene_diff['fdr_within_module'] < 0.10
module_gene_diff['gene_display_score'] = (
    module_gene_diff['abs_delta'] * module_gene_diff['det_frac_cells']
)
module_gene_diff = module_gene_diff.sort_values(['module', 'gene_display_score', 'gene'], ascending=[True, False, True]).reset_index(drop=True)
module_gene_diff.to_csv(str(MODULE_GENE_DIFF_CSV), index=False)
print()
print(f'[Saved] {MODULE_GENE_DIFF_CSV}')
print(module_gene_diff.groupby('module').size().to_string())

# ── 5) EPAS1 靶基因 × 最终代谢模块基因重叠 ────────────────────────────────────
overlap_rows = []
module_gene_map = {}

for module in MODULE_ORDER:
    tasks_present = [t for t in MODULES[module] if t in task_by_gene.index]
    if not tasks_present:
        module_gene_map[module] = []
        continue
    gene_mask = task_by_gene.loc[tasks_present].sum(axis=0) > 0
    module_genes = set(task_by_gene.columns[gene_mask].astype(str).tolist())
    module_gene_map[module] = sorted(module_genes)

    overlap = epas1_network_targets & module_genes

    mod_row = module_diff[module_diff['module'] == module]
    mod_direction = mod_row['direction'].iloc[0] if len(mod_row) > 0 else 'N/A'
    mod_delta = float(mod_row['delta_median(Dev-Norm)'].iloc[0]) if len(mod_row) > 0 else np.nan

    for gene in overlap:
        gene_rows = module_gene_diff[
            (module_gene_diff['module'] == module) &
            (module_gene_diff['gene'] == gene)
        ]
        adj_row = epas1_adj_network[epas1_adj_network['target'] == gene]
        importance = float(adj_row['importance'].iloc[0]) if len(adj_row) > 0 else np.nan
        rank_net = int(adj_row.index[0]) + 1 if len(adj_row) > 0 else 9999

        if len(gene_rows) > 0:
            gr = gene_rows.iloc[0]
            gene_delta = float(gr['delta_median(Dev-Norm)'])
            gene_fdr = float(gr['fdr_within_module'])
            gene_direction = str(gr['direction'])
            is_changed = bool(gr['is_changed_gene'])
            det_frac = float(gr['det_frac_cells'])
        else:
            gene_delta = gene_fdr = np.nan
            gene_direction = 'N/A'
            is_changed = False
            det_frac = np.nan

        concordant = (
            np.isfinite(mod_delta) and np.isfinite(gene_delta)
            and np.sign(mod_delta) == np.sign(gene_delta)
        )
        overlap_rows.append({
            'module': module,
            'module_delta': mod_delta,
            'module_direction': mod_direction,
            'gene': gene,
            'gene_delta(Dev-Norm)': gene_delta,
            'gene_fdr_within_module': gene_fdr,
            'gene_direction': gene_direction,
            'is_changed_gene': is_changed,
            'det_frac_cells': det_frac,
            'epas1_importance': importance,
            'epas1_rank_in_network': rank_net,
            'module_gene_concordant': concordant,
        })

overlap_df = pd.DataFrame(overlap_rows)
overlap_df = overlap_df.sort_values(['module', 'epas1_importance'], ascending=[True, False]).reset_index(drop=True)
overlap_df.to_csv(str(EPAS1_METAB_OVERLAP_CSV), index=False)

print()
print(f'[EPAS1 GRN 靶基因 × 最终代谢模块基因重叠汇总 | importance≥{EPAS1_MIN_IMP_NETWORK}]')
print(f'[Saved] {EPAS1_METAB_OVERLAP_CSV}')
print()

for module in MODULE_ORDER:
    sub = overlap_df[overlap_df['module'] == module]
    n_total = len(sub)
    n_changed = int(sub['is_changed_gene'].sum())
    n_concord = int(sub['module_gene_concordant'].sum())
    mod_dir = module_diff.loc[module_diff['module'] == module, 'direction'].iloc[0]
    print(f'{module}  [{mod_dir}]')
    print(f'  EPAS1 靶基因 ∩ 模块基因: {n_total} 个  |  差异显著: {n_changed}  |  方向一致: {n_concord}')
    if n_total > 0:
        show = sub.head(10)
        print(
            show[['gene', 'epas1_importance', 'gene_delta(Dev-Norm)',
                  'gene_fdr_within_module', 'is_changed_gene', 'module_gene_concordant']]
            .to_string(index=False)
        )
    print()


### 8b. 显式计算：9 条 Deviating 高通路的全部基因，以及其中受 EPAS1 网络调控的基因

上面的 `module_gene_diff / overlap_df` 是按 **4 个模块** 组织的，容易让人误以为已经单独显式提取了 **9 条 Deviating 高通路** 的全部基因。
这一节把这一步单独写出来：

- 先用 `Task_by_Gene.csv` 取出 `DEV_ELEVATED_TASKS` 对应的全部基因
- 再在 TLS 伪 bulk 层面计算这些基因的 `Normal vs Deviating` 差异
- 最后与 `EPAS1` 的 GRNBoost 网络靶基因求交


In [ ]:
DEV_ELEVATED_TASKS_EXPLICIT = [
    'Phosphatidyl-inositol synthesis',
    'Arginine synthesis',
    'Conversion of aspartate to arginine',
    'Arginine degradation',
    'Cardiolipin synthesis',
    'Triacylglycerol synthesis',
    'Synthesis of palmitoyl-CoA',
    'Phosphatidyl-serine synthesis',
    'Synthesis of glucocerebroside',
]
DEV_ELEVATED_TASK_GENE_DIFF_CSV = EPAS1_OUT_DIR / 'EPAS1_dev_elevated_task_gene_diff.csv'
DEV_ELEVATED_TASK_EPAS1_OVERLAP_CSV = EPAS1_OUT_DIR / 'EPAS1_dev_elevated_task_gene_overlap.csv'

valid_dev_tasks = [t for t in DEV_ELEVATED_TASKS_EXPLICIT if t in task_by_gene.index]
missing_dev_tasks = [t for t in DEV_ELEVATED_TASKS_EXPLICIT if t not in task_by_gene.index]
if missing_dev_tasks:
    print('[Warn] 以下 Deviating 高通路不在 Task_by_Gene 中:', missing_dev_tasks)

dev_gene_mask = task_by_gene.loc[valid_dev_tasks].sum(axis=0) > 0
dev_task_genes = [g for g in task_by_gene.columns[dev_gene_mask].astype(str).tolist() if g in tls_adata.var_names]
print(f'Deviating 高通路数: {len(valid_dev_tasks)}')
print(f'Deviating 高通路对应基因数 (Task_by_Gene ∪): {len(dev_task_genes)}')

X_dev = tls_adata[:, dev_task_genes].X
if issparse(X_dev):
    X_dev = X_dev.toarray()
X_dev = np.asarray(X_dev, dtype=np.float32)

dev_cell_expr_df = pd.DataFrame(X_dev, index=tls_adata.obs_names, columns=dev_task_genes)
dev_cell_expr_df['tls_link'] = tls_adata.obs['tls_link'].values

dev_tls_gene_df = dev_cell_expr_df.groupby('tls_link', observed=True)[dev_task_genes].mean()
dev_tls_gene_df = dev_tls_gene_df.loc[dev_tls_gene_df.index.intersection(tls_group_series.index)].copy()
dev_tls_group = tls_group_series.loc[dev_tls_gene_df.index]

dev_gene_rows = []
for gene in dev_task_genes:
    vals_n = dev_tls_gene_df.loc[dev_tls_group == 'Normal', gene].to_numpy(dtype=float)
    vals_d = dev_tls_gene_df.loc[dev_tls_group == 'Deviating', gene].to_numpy(dtype=float)
    vals_n = vals_n[np.isfinite(vals_n)]
    vals_d = vals_d[np.isfinite(vals_d)]
    if len(vals_n) == 0 or len(vals_d) == 0:
        continue
    try:
        u_stat, pval = mannwhitneyu(vals_d, vals_n, alternative='two-sided')
        rbc = (2 * u_stat / (len(vals_d) * len(vals_n))) - 1
    except Exception:
        pval, rbc = 1.0, np.nan
    med_n = float(np.median(vals_n))
    med_d = float(np.median(vals_d))
    delta = med_d - med_n
    dev_gene_rows.append({
        'gene': gene,
        'median_Normal': med_n,
        'median_Deviating': med_d,
        'delta_median(Dev-Norm)': delta,
        'rank_biserial': float(rbc) if np.isfinite(rbc) else np.nan,
        'p_value': float(pval),
        'direction': direction_text(delta),
        'present_in_epas1_network': gene in epas1_network_targets,
    })

dev_task_gene_diff = pd.DataFrame(dev_gene_rows)
dev_task_gene_diff['fdr'] = bh_fdr(dev_task_gene_diff['p_value'].to_numpy(dtype=float))
dev_task_gene_diff['dev_high'] = dev_task_gene_diff['delta_median(Dev-Norm)'] > 0
dev_task_gene_diff['dev_high_sig_fdr10'] = dev_task_gene_diff['dev_high'] & (dev_task_gene_diff['fdr'] < 0.10)
dev_task_gene_diff = dev_task_gene_diff.sort_values(
    ['present_in_epas1_network', 'delta_median(Dev-Norm)', 'gene'],
    ascending=[False, False, True]
).reset_index(drop=True)
dev_task_gene_diff.to_csv(str(DEV_ELEVATED_TASK_GENE_DIFF_CSV), index=False)

ov = dev_task_gene_diff[dev_task_gene_diff['present_in_epas1_network']].copy()
ov = ov.merge(
    epas1_adj_network[['target', 'importance']].rename(columns={'target': 'gene', 'importance': 'epas1_importance'}),
    on='gene', how='left'
).sort_values(['dev_high_sig_fdr10', 'delta_median(Dev-Norm)', 'epas1_importance'], ascending=[False, False, False]).reset_index(drop=True)
ov.to_csv(str(DEV_ELEVATED_TASK_EPAS1_OVERLAP_CSV), index=False)

print(f'[Saved] {DEV_ELEVATED_TASK_GENE_DIFF_CSV}')
print(f'[Saved] {DEV_ELEVATED_TASK_EPAS1_OVERLAP_CSV}')
print()
print('[9 条 Deviating 高通路的基因差异汇总]')
print(f'  总基因数: {len(dev_task_gene_diff)}')
print(f'  Deviating higher 基因数: {int(dev_task_gene_diff["dev_high"].sum())}')
print(f'  Deviating higher 且 FDR<0.10 基因数: {int(dev_task_gene_diff["dev_high_sig_fdr10"].sum())}')
print(f'  其中位于 EPAS1 网络中的基因数: {int(dev_task_gene_diff["present_in_epas1_network"].sum())}')
print(f'  其中 Deviating higher 且 FDR<0.10 且位于 EPAS1 网络中的基因数: {int((dev_task_gene_diff["dev_high_sig_fdr10"] & dev_task_gene_diff["present_in_epas1_network"]).sum())}')
print()
print('[Top overlap genes]')
show_cols = ['gene', 'epas1_importance', 'delta_median(Dev-Norm)', 'fdr', 'direction', 'dev_high_sig_fdr10']
print(ov[show_cols].head(20).to_string(index=False))


## 9. EPAS1 活性 × 代谢模块分数的 TLS-level 相关

In [ ]:
# 读取已有 TLS-level 代谢模块分数
module_score_df = pd.read_csv(str(TLS_METAB_MODULE_CSV), index_col=0)

# 对齐 TLS
common_tls = tls_epas1_scores.index.intersection(module_score_df.index)
common_tls = [t for t in common_tls if tls_epas1_scores.loc[t, 'tls_group'] in ['Normal', 'Deviating']]
print(f'共同 TLS: {len(common_tls)}')

epas1_aligned    = tls_epas1_scores.loc[common_tls].copy()
module_aligned   = module_score_df.loc[common_tls].copy()
group_aligned    = epas1_aligned['tls_group']

corr_rows = []
for score_col, score_label in [
    ('EPAS1_expr',        'EPAS1 RNA'),
    ('EPAS1_target_mean', 'EPAS1 target mean'),
    ('EPAS1_aucell',      'EPAS1 AUCell'),
]:
    x = epas1_aligned[score_col].to_numpy(dtype=float)
    for module in MODULE_ORDER:
        if module not in module_aligned.columns:
            continue
        y = module_aligned[module].to_numpy(dtype=float)
        finite = np.isfinite(x) & np.isfinite(y)
        if finite.sum() < 10:
            continue
        rho, pval = spearmanr(x[finite], y[finite])
        if not np.isfinite(rho):
            continue
        mod_row = module_diff[module_diff['module'] == module]
        mod_delta = float(mod_row['delta_median(Dev-Norm)'].iloc[0]) if len(mod_row) > 0 else np.nan
        epas1_delta = epas1_diff_results[score_col]['delta']
        concordant = np.sign(rho) * np.sign(mod_delta) * np.sign(epas1_delta) > 0
        corr_rows.append({
            'epas1_score':     score_col,
            'epas1_label':     score_label,
            'module':          module,
            'spearman_rho':    float(rho),
            'p_value':         float(pval),
            'module_delta':    mod_delta,
            'epas1_delta':     epas1_delta,
            'direction_concordant': bool(concordant),
            'n_tls':           int(finite.sum()),
        })

corr_df = pd.DataFrame(corr_rows)
corr_df['fdr'] = bh_fdr(corr_df['p_value'].values)
corr_df = corr_df.sort_values(['epas1_score','module']).reset_index(drop=True)
corr_df.to_csv(str(EPAS1_CORR_CSV), index=False)

print('[EPAS1 × 代谢模块 Spearman 相关]')
print(corr_df[['epas1_label','module','spearman_rho','p_value','fdr',
               'direction_concordant','module_delta','epas1_delta']].to_string(index=False))
print(f'\n[Saved] {EPAS1_CORR_CSV}')

## 10. 图3：EPAS1 × 代谢模块相关散点图

> **关于 M1 没有 M1a/M1b 拆分的说明**
>
> 旧 notebook（`两组TLS转录因子差异.ipynb`）的最后几个 cell 将 M1 拆成了：
> - M1a：PIP2→IP3 转化（任务 1）
> - M1b：磷脂酰肌醇合成（任务 2）
>
> 本 notebook 的散点图将 M1 作为整体展示（2 个任务取均值），因此只有 1 张 M1 图。
> 如需查看 M1 内部两个子任务与 EPAS1 的相关，请参见下方 **小节 15（M1 子任务拆分）**。

In [ ]:
# ── Load M1a/M1b task-level scores ────────────────────────────────────────────
_M1A_TASK = ('Conversion of 1-phosphatidyl-1D-myo-inositol 4,5-bisphosphate '
             'to 1D-myo-inositol 1,4,5-trisphosphate')
_M1B_TASK = 'Phosphatidyl-inositol synthesis'

SCAT_NORMAL = '#6E8DD8'
SCAT_DEVIATING = '#D97A73'
SCAT_MODULE_COLORS = {
    'M1: PIP2-IP3 Ca2+': '#D95C5C',
    'M2: Glycan maturation': '#4F84C4',
    'M3: Arg-Gln axis': '#D98B21',
    'M4: Lipid-Cardiolipin': '#A06CC7',
}

_task_csv = RESULTS_DIR / 'Normal_vs_Deviating_corePlusAround_tls_metabolic_task_scores.csv'
_tls_tasks = pd.read_csv(str(_task_csv), index_col=0)

_common_t = [t for t in tls_epas1_scores.index.intersection(_tls_tasks.index)
             if tls_epas1_scores.loc[t, 'tls_group'] in ['Normal', 'Deviating']]
_epas1_t = tls_epas1_scores.loc[_common_t]
_task_t = _tls_tasks.loc[_common_t]

def _subtask_scatter_data(task_name, epas1_series, task_df, group_series):
    if task_name not in task_df.columns:
        return None, None, None
    y = task_df[task_name]
    df = pd.DataFrame({'x': epas1_series, 'y': y, 'g': group_series}).dropna()
    rho, pv = spearmanr(df['x'], df['y'])
    vals_n = df.loc[df['g'] == 'Normal', 'y'].values
    vals_d = df.loc[df['g'] == 'Deviating', 'y'].values
    delta = float(np.median(vals_d) - np.median(vals_n))
    return df, rho, dict(p=pv, dir=direction_text(delta), delta=delta)

MAIN_EVIDENCE_MODULES = ['M3: Arg-Gln axis', 'M4: Lipid-Cardiolipin']

PANEL_SPECS = []
for _mod in MAIN_EVIDENCE_MODULES:
    if _mod not in module_aligned.columns:
        continue
    _df2 = pd.DataFrame({
        'x': epas1_aligned['EPAS1_aucell'],
        'y': module_aligned[_mod],
        'g': group_aligned,
    }).dropna()
    _cr = corr_df[(corr_df['epas1_score'] == 'EPAS1_aucell') & (corr_df['module'] == _mod)]
    _rho2, _p2, _fdr2, _conc = (
        _cr['spearman_rho'].iloc[0], _cr['p_value'].iloc[0],
        _cr['fdr'].iloc[0], bool(_cr['direction_concordant'].iloc[0])
    ) if len(_cr) > 0 else (np.nan, 1.0, 1.0, False)
    _mod_dir = module_diff.loc[module_diff['module'] == _mod, 'direction'].iloc[0]
    PANEL_SPECS.append(dict(
        label=_mod, bio=_mod_dir, color=SCAT_MODULE_COLORS.get(_mod, MODULE_COLORS[_mod]),
        df=_df2, rho=_rho2, p=_p2, fdr=_fdr2,
        concordant=_conc, direction=_mod_dir,
    ))

n_panels = len(PANEL_SPECS)
ncols = 2 if n_panels > 1 else 1
nrows = int(np.ceil(n_panels / ncols))
fig_w = 12.4 if ncols == 2 else 6.6
fig_h = 5.8 * nrows
fig, axes = plt.subplots(nrows, ncols, figsize=(fig_w, fig_h),
                         gridspec_kw={'hspace': 0.42, 'wspace': 0.34})
axes_flat = np.atleast_1d(axes).ravel()

from matplotlib.lines import Line2D as _L2D

for ax_i, spec in enumerate(PANEL_SPECS):
    ax = axes_flat[ax_i]
    prettify_axis(ax, grid_axis='both')
    ax.grid(axis='both', linestyle=(0, (2.2, 2.2)), linewidth=0.7,
            color='#E2E8F0', alpha=0.85, zorder=0)

    df = spec['df']
    spec_color = spec['color']
    for g, c in [('Normal', SCAT_NORMAL), ('Deviating', SCAT_DEVIATING)]:
        sub = df[df['g'] == g]
        ax.scatter(sub['x'], sub['y'], s=60, alpha=0.82, color=c,
                   edgecolors='white', linewidths=0.65, zorder=3)

    if df['x'].nunique() > 3:
        coef = np.polyfit(df['x'], df['y'], 1)
        xs = np.linspace(df['x'].min(), df['x'].max(), 200)
        ax.plot(xs, coef[0] * xs + coef[1], color=spec_color,
                lw=2.4, alpha=0.92, zorder=2)

    rho, pv, fdr_v = spec['rho'], spec['p'], spec['fdr']
    star = ('****' if pv < 1e-4 else ('***' if pv < 0.001
            else ('**' if pv < 0.01 else ('*' if pv < 0.05 else 'ns'))))
    conc_txt = 'concordant' if spec['concordant'] else 'discordant'
    ann_color = spec_color if spec['concordant'] else COLOR_MUTED
    ax.text(0.05, 0.95,
            f'ρ = {rho:+.2f}  {star}\nFDR = {fdr_v:.2e}  {conc_txt}',
            transform=ax.transAxes, ha='left', va='top', fontsize=17.1,
            color=ann_color, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.36', fc='white',
                      ec=mpl.colors.to_rgba(ann_color, 0.28),
                      lw=0.9, alpha=0.96))

    ax.text(-0.13, 1.11, chr(65 + ax_i), transform=ax.transAxes,
            fontsize=28, fontweight='bold', color=COLOR_TEXT, va='top')
    ax.set_title(spec['label'], fontsize=22.4, fontweight='bold',
                 color=spec_color, pad=12)
    add_subtitle(ax, spec['bio'], y=1.04)

    ax.set_xlabel('EPAS1 AUCell regulon score', fontsize=20.7, labelpad=6)
    ax.set_ylabel('Metabolic task/module score', fontsize=20.7, labelpad=6)
    ax.tick_params(labelsize=18)

for j in range(n_panels, len(axes_flat)):
    axes_flat[j].set_visible(False)

leg_handles = [
    _L2D([0], [0], marker='o', color='w', ms=10,
         markerfacecolor=SCAT_NORMAL, markeredgecolor='white', label='Normal TLS'),
    _L2D([0], [0], marker='o', color='w', ms=10,
         markerfacecolor=SCAT_DEVIATING, markeredgecolor='white', label='Deviating TLS'),
    _L2D([0], [0], lw=2.4, color='#6D28D9', label='Linear fit'),
]
if n_panels < len(axes_flat):
    leg_ax = axes_flat[n_panels]
    leg_ax.axis('off')
    leg_ax.add_patch(mpl.patches.FancyBboxPatch(
        (0.08, 0.24), 0.82, 0.56,
        boxstyle='round,pad=0.03,rounding_size=0.03',
        facecolor='white', edgecolor='#D1D5DB', linewidth=1.0,
        transform=leg_ax.transAxes,
    ))
    leg_ax.text(0.16, 0.68, 'Legend', transform=leg_ax.transAxes,
                fontsize=20.7, fontweight='bold', color=COLOR_TEXT)
    leg_ax.legend(handles=leg_handles, loc='center left', bbox_to_anchor=(0.14, 0.47),
                  frameon=False, fontsize=17.6)
    leg_ax.text(0.16, 0.22,
                'Each point represents one TLS\nρ and FDR are reported per panel',
                transform=leg_ax.transAxes, ha='left', va='bottom',
                fontsize=15.7, color=COLOR_MUTED, style='italic')
else:
    fig.legend(handles=leg_handles, loc='lower center', ncol=3, fontsize=17.3,
               bbox_to_anchor=(0.5, -0.04), frameon=False)

fig.suptitle('EPAS1 AUCell Activity Tracks Deviating-High TLS Metabolic Modules',
             fontsize=26, fontweight='bold', color=COLOR_TEXT, y=0.985)
fig.text(0.5, 0.945,
         'Main evidence panels restricted to the two whole modules elevated in Deviating TLS (M3/M4); M1a/M1b are shown separately below',
         ha='center', va='center', fontsize=16.5, color=COLOR_MUTED, style='italic')
fig.subplots_adjust(top=0.82, bottom=0.10, left=0.08, right=0.98)

plt.savefig(str(FIG_CORR_PDF), dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print(f'[Saved] {FIG_CORR_PDF}')
print('Note: "importance" in GRNBoost2 is a gradient-boosting feature importance score,')
print('      NOT a statistical p-value. Genes with importance≥0.05 include both')
print('      significantly and non-significantly differentially expressed genes.')
print('      Bold gene names in network figures = FDR<0.10 differential expression.')


## 11. 图3b：M1 子任务拆分 — M1a vs M1b 与 EPAS1 相关

> **M1 模块包含两条方向相反的代谢通路：**
>
> | 子任务 | 生物学意义 | 方向 |
> |--------|-----------|------|
> | **M1a** | PIP2→IP3 转化（磷脂酰肌醇 4,5-二磷酸→肌醇三磷酸） | **Normal TLS 高**（delta=−0.694, p=2.22e−07）|
> | **M1b** | 磷脂酰肌醇合成（PI synthesis） | **Deviating TLS 高**（delta=+0.036, p=4.59e−08）|
>
> 这两个子任务代表 M1 模块内部的代谢拮抗：Normal TLS 偏向 PIP2→IP3 信号传导（Ca²⁺ 释放），
> 而 Deviating TLS 偏向 PI 合成（膜脂供应）。M1 整体呈 Normal 高是两者均值的结果。

In [ ]:
FIG_M1_SPLIT_PDF = EPAS1_OUT_DIR / 'EPAS1_M1_subtask_split.pdf'

# M1 子任务定义
M1A_TASK = 'Conversion of 1-phosphatidyl-1D-myo-inositol 4,5-bisphosphate to 1D-myo-inositol 1,4,5-trisphosphate'
M1B_TASK = 'Phosphatidyl-inositol synthesis'

# ─── 读取 TLS-level 代谢 task 分数 ───────────────────────────────────────────
TLS_TASK_CSV = RESULTS_DIR / 'Normal_vs_Deviating_corePlusAround_tls_metabolic_task_scores.csv'
tls_task_df  = pd.read_csv(str(TLS_TASK_CSV), index_col=0)

# 对齐 TLS
common_tls_task = tls_epas1_scores.index.intersection(tls_task_df.index)
common_tls_task = [t for t in common_tls_task
                   if tls_epas1_scores.loc[t, 'tls_group'] in ['Normal', 'Deviating']]

epas1_task_aligned  = tls_epas1_scores.loc[common_tls_task].copy()
task_score_aligned  = tls_task_df.loc[common_tls_task].copy()
group_task_aligned  = epas1_task_aligned['tls_group']

# ─── 统计 M1a / M1b 差异 ──────────────────────────────────────────────────────
subtask_stats = {}
for label, task_name in [('M1a', M1A_TASK), ('M1b', M1B_TASK)]:
    if task_name not in task_score_aligned.columns:
        print(f'[Warn] {task_name} 不在 task_score 列中，跳过')
        continue
    vals_n = task_score_aligned.loc[group_task_aligned == 'Normal',    task_name].dropna().to_numpy()
    vals_d = task_score_aligned.loc[group_task_aligned == 'Deviating', task_name].dropna().to_numpy()
    _, pv  = mannwhitneyu(vals_n, vals_d, alternative='two-sided')
    delta  = float(np.median(vals_d) - np.median(vals_n))
    rbc    = rankbiserial(vals_n, vals_d)
    subtask_stats[label] = dict(task=task_name, delta=delta, p=pv, rbc=rbc,
                                n_N=len(vals_n), n_D=len(vals_d))
    print(f'{label}: delta={delta:+.3f}, p={pv:.2e}, rbc={rbc:+.3f}  [{direction_text(delta)}]')

# ─── 计算 EPAS1 × 子任务 Spearman 相关 ─────────────────────────────────────────
for label, task_name in [('M1a', M1A_TASK), ('M1b', M1B_TASK)]:
    if task_name not in task_score_aligned.columns:
        continue
    x = epas1_task_aligned['EPAS1_aucell'].to_numpy(dtype=float)
    y = task_score_aligned[task_name].to_numpy(dtype=float)
    finite = np.isfinite(x) & np.isfinite(y)
    rho, pv = spearmanr(x[finite], y[finite])
    subtask_stats[label]['rho'] = rho
    subtask_stats[label]['rho_p'] = pv
    print(f'{label} × EPAS1 AUCell: ρ={rho:.3f}, p={pv:.2e}')

# ─── 找出每个子任务中的 EPAS1 靶基因 ─────────────────────────────────────────────
m1a_genes_module = set(task_by_gene.columns[task_by_gene.loc[M1A_TASK] > 0]) if M1A_TASK in task_by_gene.index else set()
m1b_genes_module = set(task_by_gene.columns[task_by_gene.loc[M1B_TASK] > 0]) if M1B_TASK in task_by_gene.index else set()

m1a_overlap = epas1_network_targets & m1a_genes_module
m1b_overlap = epas1_network_targets & m1b_genes_module
print(f'\nM1a 模块基因数: {len(m1a_genes_module)} | EPAS1 靶基因重叠: {len(m1a_overlap)} → {sorted(m1a_overlap)}')
print(f'M1b 模块基因数: {len(m1b_genes_module)} | EPAS1 靶基因重叠: {len(m1b_overlap)} → {sorted(m1b_overlap)}')

In [ ]:
FIG_M1_SPLIT_PDF = EPAS1_OUT_DIR / 'EPAS1_M1_subtask_split.pdf'

# M1 sub-task configs
subtask_configs = [
    ('M1a', M1A_TASK, COLOR_NORMAL,    'Normal higher',    'PIP2→IP3 conversion\n(Ca²⁺ signaling)'),
    ('M1b', M1B_TASK, COLOR_DEVIATING, 'Deviating higher', 'PI synthesis\n(membrane lipid supply)'),
]

# ── Wide figure: 5 panels in 1 row, generous width ───────────────────────────
fig = plt.figure(figsize=(26, 6))
gs = fig.add_gridspec(1, 5, wspace=0.44,
                      width_ratios=[1.1, 1.1, 1.4, 1.4, 1.6])

ax_v1  = fig.add_subplot(gs[0, 0])
ax_v2  = fig.add_subplot(gs[0, 1])
ax_s1  = fig.add_subplot(gs[0, 2])
ax_s2  = fig.add_subplot(gs[0, 3])
ax_bar = fig.add_subplot(gs[0, 4])

# ── Violin panels ─────────────────────────────────────────────────────────────
for ax_v, (label, task_name, task_color, direction_label, bio_label) in zip(
        [ax_v1, ax_v2], subtask_configs):
    prettify_axis(ax_v, grid_axis='y')
    if task_name not in task_score_aligned.columns:
        ax_v.set_visible(False)
        continue
    vals_n = task_score_aligned.loc[group_task_aligned=='Normal',    task_name].dropna().to_numpy()
    vals_d = task_score_aligned.loc[group_task_aligned=='Deviating', task_name].dropna().to_numpy()

    parts = ax_v.violinplot([vals_n, vals_d], positions=[0, 1],
                             showmedians=True, showextrema=False, widths=0.65)
    for pc, c in zip(parts['bodies'], [COLOR_NORMAL, COLOR_DEVIATING]):
        pc.set_facecolor(c); pc.set_alpha(0.50); pc.set_edgecolor('white')
    parts['cmedians'].set_color('black'); parts['cmedians'].set_linewidth(2.5)

    # IQR bar
    for xi, (vals, c) in enumerate(zip([vals_n, vals_d], [COLOR_NORMAL, COLOR_DEVIATING])):
        q1, q3 = np.percentile(vals, [25, 75])
        ax_v.vlines(xi, q1, q3, lw=5, color='white', zorder=4)
        ax_v.vlines(xi, q1, q3, lw=3.2, color=c, alpha=0.85, zorder=5)

    # Jitter
    rng = np.random.default_rng(42)
    for xi, (vals, c) in enumerate(zip([vals_n, vals_d], [COLOR_NORMAL, COLOR_DEVIATING])):
        jitter = rng.uniform(-0.09, 0.09, len(vals))
        ax_v.scatter(xi+jitter, vals, s=22, color=c, alpha=0.55,
                     edgecolors='white', linewidths=0.3, zorder=3)

    # Significance bar
    st   = subtask_stats.get(label, {})
    pval = st.get('p', 1.0)
    star = ('****' if pval<1e-4 else ('***' if pval<0.001
            else ('**' if pval<0.01 else ('*' if pval<0.05 else 'ns'))))
    y_all   = np.concatenate([vals_n, vals_d])
    y_span  = y_all.max() - y_all.min()
    y_line  = y_all.max() + y_span * 0.10
    ax_v.plot([0,1], [y_line, y_line], color='black', lw=1.2)
    ax_v.plot([0,0], [y_all.max()+y_span*0.02, y_line], color='black', lw=1.0)
    ax_v.plot([1,1], [y_all.max()+y_span*0.02, y_line], color='black', lw=1.0)
    ax_v.text(0.5, y_line + y_span*0.013,
              f'{star}  p={pval:.2e}',
              ha='center', va='bottom', fontsize=18.7, fontweight='bold', color=COLOR_TEXT)

    ax_v.set_xticks([0, 1])
    ax_v.set_xticklabels(['Normal', 'Deviating'], fontsize=20, fontweight='bold')
    ax_v.tick_params(axis='x', length=0)
    ax_v.set_ylabel('Task score', fontsize=20)
    ax_v.set_xlim(-0.6, 1.6)
    ax_v.set_title(f'{label}\n{bio_label}', fontsize=21.3, fontweight='bold',
                   color=task_color, pad=12)
    add_subtitle(ax_v, f'{direction_label}  |  RBC={st.get("rbc",0):+.2f}', y=1.02)

# ── Scatter panels ────────────────────────────────────────────────────────────
for ax_s, (label, task_name, task_color, direction_label, bio_label) in zip(
        [ax_s1, ax_s2], subtask_configs):
    prettify_axis(ax_s, grid_axis='both')
    if task_name not in task_score_aligned.columns:
        ax_s.set_visible(False)
        continue

    sdf = pd.DataFrame({
        'x': epas1_task_aligned['EPAS1_aucell'],
        'y': task_score_aligned[task_name],
        'g': group_task_aligned,
    }).dropna()

    for g, c, m in [('Normal', COLOR_NORMAL, 'o'), ('Deviating', COLOR_DEVIATING, 'D')]:
        sub = sdf[sdf['g']==g]
        ax_s.scatter(sub['x'], sub['y'], s=52, alpha=0.72, color=c,
                     marker=m, edgecolors='white', linewidths=0.5, label=g, zorder=3)

    if sdf['x'].nunique() > 3:
        coef = np.polyfit(sdf['x'], sdf['y'], 1)
        xs   = np.linspace(sdf['x'].min(), sdf['x'].max(), 200)
        ax_s.plot(xs, coef[0]*xs+coef[1], color=COLOR_EPAS1, lw=2.2, ls='--', alpha=0.85)

    st   = subtask_stats.get(label, {})
    rho, pv = st.get('rho', np.nan), st.get('rho_p', 1.0)
    star = ('****' if pv<1e-4 else ('***' if pv<0.001
            else ('**' if pv<0.01 else ('*' if pv<0.05 else 'ns'))))
    ax_s.text(0.04, 0.96,
              f'ρ = {rho:+.2f}  {star}\np = {pv:.2e}',
              transform=ax_s.transAxes, ha='left', va='top', fontsize=17.3,
              color=COLOR_EPAS1, fontweight='bold',
              bbox=dict(boxstyle='round,pad=0.32', fc='white', ec='#D1D5DB', lw=0.7, alpha=0.95))
    ax_s.set_xlabel('EPAS1 AUCell score', fontsize=20, labelpad=6)
    ax_s.set_ylabel('Task score', fontsize=20, labelpad=6)
    ax_s.set_title(f'{label} × EPAS1\n{bio_label}', fontsize=21.3,
                   fontweight='bold', color=task_color, pad=12)
    add_subtitle(ax_s, direction_label, y=1.02)
    ax_s.tick_params(labelsize=17.3)
    if ax_s is ax_s1:
        ax_s.legend(fontsize=17.3, markerscale=1.2, frameon=True,
                    edgecolor='#D1D5DB', framealpha=0.9)

# ── Bar chart: M1 gene expression directions ──────────────────────────────────
prettify_axis(ax_bar, grid_axis='x')

m1_overlap_df = overlap_df[overlap_df['module'] == 'M1: PIP2-IP3 Ca2+'].copy()
m1_overlap_df = m1_overlap_df.sort_values('epas1_importance', ascending=True)

subtask_palette = {'M1a': COLOR_NORMAL, 'M1b': COLOR_DEVIATING,
                   'M1a+M1b': '#9333EA', 'M1 (other)': COLOR_NEUTRAL}

if len(m1_overlap_df) > 0:
    def _get_m1_bar_label(gene):
        a = gene in m1a_overlap; b = gene in m1b_overlap
        return 'M1a+M1b' if (a and b) else ('M1a' if a else ('M1b' if b else 'M1 (other)'))

    m1_overlap_df['subtask']  = m1_overlap_df['gene'].map(_get_m1_bar_label)
    bar_colors = m1_overlap_df['subtask'].map(subtask_palette).tolist()
    genes_bar  = m1_overlap_df['gene'].tolist()
    deltas_bar = m1_overlap_df['gene_delta(Dev-Norm)'].tolist()
    fdr_vals   = m1_overlap_df['gene_fdr_within_module'].tolist()

    y_pos = np.arange(len(genes_bar))
    ax_bar.barh(y_pos, deltas_bar, color=bar_colors,
                height=0.62, edgecolor='white', linewidth=0.5)
    ax_bar.axvline(0, color='#94A3B8', lw=0.9, ls='--')

    ax_bar.set_yticks(y_pos)
    ax_bar.set_yticklabels(genes_bar, fontsize=17.3, fontstyle='italic')
    # Bold ticks for significant genes
    for ytick, fdr_v in zip(ax_bar.get_yticklabels(), fdr_vals):
        if np.isfinite(fdr_v) and fdr_v < 0.10:
            ytick.set_fontweight('bold')

    # FDR significance markers
    for i, (delta_v, fdr_v) in enumerate(zip(deltas_bar, fdr_vals)):
        if np.isfinite(fdr_v) and fdr_v < 0.05:
            ax_bar.text(delta_v + 0.005 * np.sign(delta_v if delta_v != 0 else 1),
                        i, '*', ha='left', va='center',
                        fontsize=20, fontweight='bold', color=COLOR_TEXT)

    ax_bar.set_xlabel('Delta median\n(Deviating − Normal)', fontsize=20, labelpad=6)
    ax_bar.set_title('M1 gene expression direction\n(EPAS1 GRN targets, importance≥0.05)',
                     fontsize=21.3, fontweight='bold',
                     color=MODULE_COLORS['M1: PIP2-IP3 Ca2+'], pad=12)
    ax_bar.tick_params(labelsize=17.3)

    legend_patches = [
        mpatches.Patch(color=COLOR_NORMAL,    label='M1a only — PIP2→IP3'),
        mpatches.Patch(color=COLOR_DEVIATING, label='M1b only — PI synthesis'),
        mpatches.Patch(color='#9333EA',       label='M1a + M1b'),
        mpatches.Patch(color=COLOR_NEUTRAL,   label='M1 (task unassigned)'),
    ]
    ax_bar.legend(handles=legend_patches, fontsize=14.7, loc='lower right',
                  frameon=True, edgecolor='#D1D5DB')

fig.suptitle(
    'M1 internal antagonism: M1a (PIP2→IP3, Normal higher) vs M1b (PI synthesis, Deviating higher)\n'
    'EPAS1 AUCell score separately correlates with each sub-task',
    fontsize=24, fontweight='bold', color=COLOR_TEXT, y=1.03,
)
plt.savefig(str(FIG_M1_SPLIT_PDF), dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print(f'[Saved] {FIG_M1_SPLIT_PDF}')

## 11. 图4：EPAS1 → 代谢模块基因 二分图网络（主图仅保留 M3 / M4）

> **主图策略调整**
>
> M1 模块内部方向相反：`M1a = Normal higher`，`M1b = Deviating higher`，因此 **不再把 M1 放进支持 EPAS1 的主证据图**。
> 主图只保留 **整体在 Deviating TLS 中升高** 且与 EPAS1 更一致的两个模块：`M3` 和 `M4`。
> `M1a / M1b` 的拆分结果保留在单独的补充图中解释。

> **绘图规则**
>
> - 主图默认只展示 `module_gene_concordant=True` 的基因
> - 基因名**加粗** = `FDR within module < 0.10`
> - 基因后 `* / ** / ***` = 更严格的 `FDR < 0.05 / 0.01 / 0.001`
> - 侧点颜色：蓝 = `Normal higher`，红 = `Deviating higher`


In [ ]:
SHOW_ALL_CONCORDANT_GENES_NET = True
TOP_GENES_PER_MODULE_NET = 10
FIG_NETWORK_V2_PDF = EPAS1_OUT_DIR / 'EPAS1_metabolic_gene_network_v2_all_targets.pdf'
NETWORK_ONLY_CONCORDANT = True   # 主图只展示与模块方向一致的基因，去掉虚线/相反方向基因

NET_COLOR_NORMAL = '#5B8FD9'
NET_COLOR_DEVIATING = '#EE8C6E'
NET_COLOR_EPAS1 = '#6D28D9'
NET_MODULE_COLORS = {
    'M1: PIP2-IP3 Ca2+': '#D95C5C',
    'M2: Glycan maturation': '#4F84C4',
    'M3: Arg-Gln axis': '#D98B21',
    'M4: Lipid-Cardiolipin': '#A06CC7',
}

_M1A_TASK_NET = ('Conversion of 1-phosphatidyl-1D-myo-inositol 4,5-bisphosphate '
                 'to 1D-myo-inositol 1,4,5-trisphosphate')
_M1B_TASK_NET = 'Phosphatidyl-inositol synthesis'
_m1a_genes_net = (set(task_by_gene.columns[task_by_gene.loc[_M1A_TASK_NET] > 0].tolist())
                  if _M1A_TASK_NET in task_by_gene.index else set())
_m1b_genes_net = (set(task_by_gene.columns[task_by_gene.loc[_M1B_TASK_NET] > 0].tolist())
                  if _M1B_TASK_NET in task_by_gene.index else set())

def _get_m1_subtask(gene):
    a = gene in _m1a_genes_net
    b = gene in _m1b_genes_net
    if a and b:
        return '[a+b]'
    if a:
        return '[a]'
    if b:
        return '[b]'
    return ''

def _network_gene_ys(n_genes):
    if n_genes <= 0:
        return np.array([])
    if n_genes == 1:
        return np.array([0.50])
    if n_genes == 2:
        return np.array([0.60, 0.40])
    if n_genes == 3:
        return np.array([0.67, 0.50, 0.33])
    if n_genes <= 6:
        return np.linspace(0.70, 0.30, n_genes)
    if n_genes <= 9:
        return np.linspace(0.73, 0.24, n_genes)
    return np.linspace(0.76, 0.20, n_genes)

MAIN_EVIDENCE_MODULES_NET = ['M3: Arg-Gln axis', 'M4: Lipid-Cardiolipin']

active_modules = []
plot_data_cache = {}
for module in MAIN_EVIDENCE_MODULES_NET:
    sub_all = overlap_df[overlap_df['module'] == module].copy()
    if sub_all.empty:
        continue

    sub_plot = sub_all.copy()
    if NETWORK_ONLY_CONCORDANT:
        sub_plot = sub_plot[sub_plot['module_gene_concordant']].copy()
    if sub_plot.empty:
        continue

    sub_plot = sub_plot.sort_values(
        ['epas1_importance', 'is_changed_gene', 'gene'],
        ascending=[False, False, True]
    ).reset_index(drop=True)
    if not SHOW_ALL_CONCORDANT_GENES_NET:
        sub_plot = sub_plot.head(TOP_GENES_PER_MODULE_NET).copy()

    active_modules.append(module)
    plot_data_cache[module] = {'plot': sub_plot, 'all': sub_all}

if not active_modules:
    print('[Warn] No module-gene overlaps found')
else:
    panel_units = [max(0.95, 0.18 * len(plot_data_cache[m]['plot']) + 0.76) for m in active_modules]
    n_panels = len(active_modules)
    fig_h = 1.24 * sum(panel_units) + 0.82

    fig, axes = plt.subplots(
        n_panels, 1,
        figsize=(8.2, fig_h),
        gridspec_kw={'hspace': 0.09, 'height_ratios': panel_units}
    )
    if n_panels == 1:
        axes = [axes]

    for ax, module in zip(axes, active_modules):
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.axis('off')

        mod_color = NET_MODULE_COLORS.get(module, MODULE_COLORS.get(module, '#64748B'))
        gene_data = plot_data_cache[module]['plot']
        n_genes = len(gene_data)

        box_x = 0.032
        box_w = 0.802
        box_right = box_x + box_w
        ax.add_patch(mpl.patches.FancyBboxPatch(
            (box_x, 0.040), box_w, 0.914,
            boxstyle='round,pad=0.012,rounding_size=0.022',
            transform=ax.transAxes,
            facecolor=mpl.colors.to_rgba(mod_color, 0.040),
            edgecolor=mpl.colors.to_rgba(mod_color, 0.22),
            linewidth=1.18, zorder=0,
        ))

        left_dot_x = 0.155
        right_dot_x = 0.425
        left_text_x = 0.132
        right_text_x = 0.500
        left_status_x = 0.178
        right_status_x = 0.402
        imp_bar_x0 = 0.242
        imp_bar_maxw = 0.108
        val_text_x = imp_bar_x0 + imp_bar_maxw + 0.012

        epas1_y = 0.50
        gene_ys = _network_gene_ys(n_genes)
        max_imp = max(float(gene_data['epas1_importance'].max()), 1e-6)
        is_m1 = module == 'M1: PIP2-IP3 Ca2+'

        for (_, row), gy in zip(gene_data.iterrows(), gene_ys):
            imp = row['epas1_importance']
            lw = 0.5 + 1.75 * imp / max_imp
            alpha = 0.26 + 0.68 * imp / max_imp
            ax.plot([left_dot_x + 0.018, right_dot_x - 0.018], [epas1_y, gy],
                    color=mod_color, lw=lw, alpha=alpha,
                    solid_capstyle='round', zorder=1)

        # (epas1_side_col removed — status dots no longer drawn)
        ax.scatter([left_dot_x], [epas1_y], s=500,
                   color=mpl.colors.to_rgba(NET_COLOR_EPAS1, 0.10), zorder=3)
        ax.scatter([left_dot_x], [epas1_y], s=270, color=NET_COLOR_EPAS1,
                   edgecolor='white', linewidth=1.65, marker='*', zorder=5)
        # (removed EPAS1 status dot)
        ax.text(left_text_x, epas1_y, 'EPAS1',
                ha='right', va='center', fontsize=20.8, fontstyle='italic',
                fontweight='bold', color=NET_COLOR_EPAS1, zorder=7)

        for (_, row), gy in zip(gene_data.iterrows(), gene_ys):
            gene = row['gene']
            changed = row['is_changed_gene']
            gd = row.get('gene_delta(Dev-Norm)', np.nan)
            gfdr = row.get('gene_fdr_within_module', np.nan)
            imp = row['epas1_importance']
            # (gene_side_col removed — status dots no longer drawn)

            bar_w = imp_bar_maxw * imp / max_imp
            ax.add_patch(mpl.patches.FancyBboxPatch(
                (imp_bar_x0, gy - 0.017), bar_w, 0.034,
                boxstyle='round,pad=0.0024',
                facecolor=mpl.colors.to_rgba(mod_color, 0.66),
                edgecolor='none', zorder=2,
            ))
            ax.scatter([right_dot_x], [gy],
                       s=68 if changed else 50,
                       facecolor='white', edgecolor=mod_color,
                       linewidth=1.95 if changed else 0.92, zorder=4)
            # (removed gene status dot)
            ax.text(val_text_x, gy, f'{imp:.2f}',
                    ha='left', va='center', fontsize=11.5,
                    color=COLOR_MUTED, zorder=6)

            star_str = (' ***' if (np.isfinite(gfdr) and gfdr < 0.001)
                        else (' **' if (np.isfinite(gfdr) and gfdr < 0.01)
                        else (' *' if (np.isfinite(gfdr) and gfdr < 0.05) else '')))
            m1_tag = f' {_get_m1_subtask(gene)}' if is_m1 else ''
            ax.text(right_text_x, gy, f'{gene}{m1_tag}{star_str}',
                    ha='left', va='center', fontsize=15.1,
                    fontstyle='italic',
                    fontweight='bold' if changed else 'normal',
                    color=COLOR_TEXT, zorder=6)

        mod_dir = module_diff.loc[module_diff['module'] == module, 'direction'].iloc[0]
        corr_row = corr_df[(corr_df['epas1_score'] == 'EPAS1_aucell') & (corr_df['module'] == module)]
        rho_text = f'ρ={corr_row["spearman_rho"].iloc[0]:+.2f}, p={corr_row["p_value"].iloc[0]:.1e}'
        n_changed = int(gene_data['is_changed_gene'].sum())

        ax.text(box_x + 0.010, 0.950, module,
                ha='left', va='center', fontsize=20.3, fontweight='bold',
                color=mod_color, zorder=8)
        ax.text(box_x + 0.010, 0.892, mod_dir,
                ha='left', va='center', fontsize=11.9,
                color=COLOR_MUTED, style='italic', zorder=8)
        # (removed FDR/rho subtitle)

    fig.suptitle(
        'EPAS1 Direct GRN Links to Deviating-High Metabolic Targets',
        fontsize=18.7, fontweight='bold', color=COLOR_TEXT, y=0.992,
    )
    fig.text(0.5, 0.966,
             'Main network restricted to concordant targets from the two whole modules elevated in Deviating TLS (M3/M4); M1 is shown separately in the split figure',
             ha='center', va='center', fontsize=13.4, color=COLOR_MUTED, style='italic')

    fig.text(0.38, 0.012, 'bold = FDR<0.10', ha='left', va='bottom',
             fontsize=11.5, color=COLOR_MUTED)
    fig.text(0.56, 0.012, 'Bar & number = GRNBoost2 importance score', ha='left', va='bottom',
             fontsize=11.5, color=COLOR_MUTED)



    fig.subplots_adjust(top=0.94, bottom=0.045, left=0.05, right=0.985, hspace=0.09)
    plt.savefig(str(FIG_NETWORK_V2_PDF), dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()
    print(f'[Saved] {FIG_NETWORK_V2_PDF}')

print('\n[Overlap summary — EPAS1 GRN targets (importance≥0.05) × metabolic module genes]')
for module in MODULE_ORDER:
    sub = overlap_df[overlap_df['module'] == module]
    mod_dir = module_diff.loc[module_diff['module'] == module, 'direction'].iloc[0]
    print(f'  {module} [{mod_dir}]: {len(sub)} overlapping genes'
          f' | concordant: {int(sub["module_gene_concordant"].sum())}'
          f' | sig. changed: {int(sub["is_changed_gene"].sum())}'
          f' | top genes: {sub.head(5)["gene"].tolist()}')


## 12. 图5：EPAS1 关键代谢靶基因在 Normal vs Deviating TLS 的表达热图

In [ ]:
GENES_PER_MODULE_HEATMAP = 6
FIG_HEATMAP_V2_PDF = EPAS1_OUT_DIR / 'EPAS1_target_gene_heatmap_v2.pdf'

# ── Select top genes per module ───────────────────────────────────────────────
heatmap_gene_list   = []
heatmap_gene_module = {}
MAIN_EVIDENCE_MODULES_HEATMAP = ['M3: Arg-Gln axis', 'M4: Lipid-Cardiolipin']
for module in MAIN_EVIDENCE_MODULES_HEATMAP:
    sub = (overlap_df[overlap_df['module'] == module]
           .sort_values('epas1_importance', ascending=False)
           .head(GENES_PER_MODULE_HEATMAP))
    for gene in sub['gene'].tolist():
        if gene not in heatmap_gene_list and gene in tls_adata.var_names:
            heatmap_gene_list.append(gene)
            heatmap_gene_module[gene] = module

# Prepend EPAS1 itself
if 'EPAS1' in tls_adata.var_names and 'EPAS1' not in heatmap_gene_list:
    heatmap_gene_list.insert(0, 'EPAS1')
    heatmap_gene_module['EPAS1'] = '_EPAS1_TF_'

heatmap_genes_avail = [g for g in heatmap_gene_list if g in tls_adata.var_names]
print(f'Heatmap genes: {len(heatmap_genes_avail)}')

if len(heatmap_genes_avail) == 0:
    print('[Warn] No genes available for heatmap')
else:
    # ── Build TLS-level mean expression matrix ────────────────────────────────
    X_hm = tls_adata[:, heatmap_genes_avail].X
    if issparse(X_hm):
        X_hm = X_hm.toarray()
    X_hm = np.asarray(X_hm, dtype=np.float32)

    hm_cell_df = pd.DataFrame(X_hm, index=tls_adata.obs_names,
                               columns=heatmap_genes_avail)
    hm_cell_df['tls_link']  = tls_adata.obs['tls_link'].values
    hm_cell_df['tls_group'] = tls_adata.obs['tls_group'].values

    hm_tls_df    = hm_cell_df.groupby('tls_link', observed=True)[heatmap_genes_avail].mean()
    hm_tls_group = hm_cell_df.groupby('tls_link', observed=True)['tls_group'].agg(majority_vote)
    hm_tls_df['tls_group'] = hm_tls_group
    hm_tls_df = (hm_tls_df[hm_tls_df['tls_group'].isin(['Normal', 'Deviating'])]
                 .sort_values('tls_group', ascending=False))

    hm_mat  = hm_tls_df[heatmap_genes_avail].to_numpy(dtype=float)
    g_mean  = hm_mat.mean(axis=0)
    g_std   = hm_mat.std(axis=0); g_std[g_std == 0] = 1.0
    hm_z    = (hm_mat - g_mean) / g_std

    group_colors = [COLOR_DEVIATING if g == 'Deviating' else COLOR_NORMAL
                    for g in hm_tls_df['tls_group']]

    # ── Figure ────────────────────────────────────────────────────────────────
    fig_w = max(12, 0.78 * len(heatmap_genes_avail) + 2.5)
    fig_h = max(7,  0.20 * len(hm_tls_df) + 3.0)
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    ax.set_facecolor('white')

    cmap = sns.diverging_palette(220, 15, s=88, l=48, as_cmap=True)
    sns.heatmap(
        hm_z,
        cmap=cmap, center=0, vmin=-2.5, vmax=2.5,
        linewidths=0.28, linecolor='white',
        xticklabels=heatmap_genes_avail,
        yticklabels=False, ax=ax,
        cbar_kws={'label': 'Z-score (per gene)', 'shrink': 0.52,
                  'pad': 0.015, 'aspect': 28},
    )

    # Row-side color bar (group)
    ax.yaxis.set_visible(True)
    for i, color in enumerate(group_colors):
        ax.add_patch(mpl.patches.Rectangle(
            (-0.72, i), 0.56, 1,
            transform=ax.get_yaxis_transform(),
            color=color, alpha=0.80, clip_on=False,
        ))

    # Module color separator lines + module labels above heatmap
    gene_positions = {g: i for i, g in enumerate(heatmap_genes_avail)}
    module_boundaries = []   # (start_idx, end_idx, module)
    prev_mod, start_i = None, 0
    for i, gene in enumerate(heatmap_genes_avail):
        mod = heatmap_gene_module.get(gene, '')
        if mod != prev_mod:
            if prev_mod is not None:
                module_boundaries.append((start_i, i - 1, prev_mod))
            start_i  = i
            prev_mod = mod
    module_boundaries.append((start_i, len(heatmap_genes_avail) - 1, prev_mod))

    for (s, e, mod) in module_boundaries:
        if mod not in MODULE_COLORS:
            continue
        mc = MODULE_COLORS[mod]
        mid_x = (s + e + 1) / 2
        ax.axvline(s, color=mc, lw=1.8, alpha=0.5, zorder=5)
        # Module label above x-axis
        ax.text(mid_x, len(hm_tls_df) + len(hm_tls_df) * 0.045,
                mod.split(':')[0],   # e.g. "M4"
                ha='center', va='bottom', fontsize=12.7,
                color=mc, fontweight='bold',
                transform=ax.get_xaxis_transform() if False else ax.transData,
                rotation=0)

    # X-tick labels: italic gene names, colored by module, bold = significant
    ax.set_xticklabels(ax.get_xticklabels(), rotation=42, ha='right',
                       fontsize=14, fontstyle='italic')
    for tick, gene in zip(ax.get_xticklabels(), heatmap_genes_avail):
        mod = heatmap_gene_module.get(gene, '')
        if gene == 'EPAS1':
            tick.set_color(COLOR_EPAS1); tick.set_fontweight('bold')
        else:
            tick.set_color(MODULE_COLORS.get(mod, COLOR_TEXT))
            gene_info = overlap_df[overlap_df['gene'] == gene]
            if len(gene_info) > 0 and bool(gene_info['is_changed_gene'].iloc[0]):
                tick.set_fontweight('bold')

    # Color-bar label size
    cbar = ax.collections[0].colorbar
    cbar.ax.tick_params(labelsize=13.3)
    cbar.set_label('Z-score (per gene)', fontsize=14.7)

    # Legend
    legend_handles = [
        mpatches.Patch(color=COLOR_NORMAL,    label='Normal TLS'),
        mpatches.Patch(color=COLOR_DEVIATING, label='Deviating TLS'),
    ] + [
        mpatches.Patch(color=MODULE_COLORS[m], label=m)
        for m in MAIN_EVIDENCE_MODULES_HEATMAP if m in set(heatmap_gene_module.values())
    ] + [
        mpatches.Patch(color=COLOR_EPAS1, label='EPAS1 (TF)'),
    ]
    ax.legend(handles=legend_handles, loc='upper left', fontsize=12,
              bbox_to_anchor=(1.13, 1.0), ncol=1, frameon=True,
              edgecolor='#D1D5DB', framealpha=0.95,
              title='Group / Module', title_fontsize=17.7)

    ax.set_title(
        'EPAS1 and its direct GRN target genes: TLS-level expression heatmap (Z-score)\n'
        f'Top-{GENES_PER_MODULE_HEATMAP} EPAS1 targets per module (importance ≥ 0.05)'
        ' │ Bold gene name = FDR<0.10 significantly changed',
        fontsize=17.3, fontweight='bold', pad=20, color=COLOR_TEXT,
    )
    ax.set_xlabel('Gene', fontsize=17.3, labelpad=10)

    plt.tight_layout()
    plt.savefig(str(FIG_HEATMAP_V2_PDF), dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()
    print(f'[Saved] {FIG_HEATMAP_V2_PDF}')

## 13. 综合证据汇总

In [ ]:
print('=' * 72)
print('EPAS1 作为 TLS 代谢差异关键 TF 的综合证据汇总')
print('=' * 72)

# 1. pySCENIC 过滤原因
print('\n[1] pySCENIC CTX 过滤原因')
print(f'  • EPAS1 在 GRN（adjacencies）中有 {len(epas1_adj)} 个靶基因 ✓')
print(f'  • 但 CTX motif pruning 步骤中 EPAS1 被过滤（HIF-2α/HRE motif 在数据库中支持不足）')
print(f'  • 本分析通过手动构建 regulon（top-{EPAS1_TOP_N_TARGETS} adjacency 靶基因）绕过此限制')

# 2. EPAS1 自身表达差异
print('\n[2] EPAS1 自身表达 Normal vs Deviating TLS')
r = epas1_diff_results['EPAS1_expr']
print(f'  • Normal 中位数: {r["median_Normal"]:.4f}')
print(f'  • Deviating 中位数: {r["median_Deviating"]:.4f}')
print(f'  • Delta (Dev-Norm): {r["delta"]:+.4f}')
print(f'  • rank-biserial: {r["rank_biserial"]:+.3f}')
print(f'  • p-value: {r["p_value"]:.2e}')
print(f'  • 方向: {r["direction"]}')

# 3. 手动 regulon 分数差异
print('\n[3] EPAS1 手动 regulon (AUCell-style) 差异')
r = epas1_diff_results['EPAS1_aucell']
print(f'  • Normal 中位数: {r["median_Normal"]:.4f}')
print(f'  • Deviating 中位数: {r["median_Deviating"]:.4f}')
print(f'  • Delta: {r["delta"]:+.4f}')
print(f'  • rank-biserial: {r["rank_biserial"]:+.3f}')
print(f'  • p-value: {r["p_value"]:.2e}')
print(f'  • 方向: {r["direction"]}')

# 4. EPAS1 × 代谢模块相关
print('\n[4] EPAS1 AUCell 分 × 代谢模块分数 Spearman 相关')
corr_sub = corr_df[corr_df['epas1_score'] == 'EPAS1_aucell'].sort_values('spearman_rho', ascending=False)
print(corr_sub[['module','spearman_rho','p_value','fdr','direction_concordant']].to_string(index=False))

# 5. 代谢基因直接证据（使用网络分析门槛的靶基因集，即 importance≥EPAS1_MIN_IMP_NETWORK）
print(f'\n[5] EPAS1 直接靶基因（GRN, importance≥{EPAS1_MIN_IMP_NETWORK}）× 代谢模块基因 重叠')
for module in MODULE_ORDER:
    sub = overlap_df[overlap_df['module'] == module]
    n_total   = len(sub)
    n_changed = int(sub['is_changed_gene'].sum())
    n_concord = int(sub['module_gene_concordant'].sum())
    changed_conc = sub[sub['is_changed_gene'] & sub['module_gene_concordant']]
    mod_dir   = module_diff.loc[module_diff['module']==module, 'direction'].iloc[0]
    print(f'  {module} [{mod_dir}]')
    print(f'    重叠基因: {n_total} | 显著变化: {n_changed} | 方向一致: {n_concord}')
    if not changed_conc.empty:
        print(f'    关键基因: {changed_conc["gene"].head(6).tolist()}')

print('\n[6] 输出文件')
fig_network_out = FIG_NETWORK_V2_PDF if 'FIG_NETWORK_V2_PDF' in globals() else FIG_NETWORK_PDF
fig_heatmap_out = FIG_HEATMAP_V2_PDF if 'FIG_HEATMAP_V2_PDF' in globals() else FIG_HEATMAP_PDF
extra_figs = [FIG_M1_SPLIT_PDF] if 'FIG_M1_SPLIT_PDF' in globals() else []
for fpath in [
    EPAS1_REGULON_CSV, EPAS1_TLS_SCORE_CSV, EPAS1_METAB_OVERLAP_CSV,
    EPAS1_CORR_CSV, FIG_VOLCANO_PDF, FIG_SCORE_PDF,
    FIG_CORR_PDF, fig_network_out, fig_heatmap_out,
] + extra_figs:
    exists = '✓' if Path(fpath).exists() else '✗'
    print(f'  {exists} {fpath}')

print('\n[结论]')
epas1_auc_dir = epas1_diff_results['EPAS1_aucell']['direction']
print(f'  EPAS1 regulon 活性在 TLS 中 {epas1_auc_dir}')
concord_modules = corr_df[
    (corr_df['epas1_score'] == 'EPAS1_aucell') &
    corr_df['direction_concordant']
]['module'].tolist()
print(f'  与 EPAS1 活性方向一致的代谢模块: {concord_modules}')
print('  主证据图仅保留整体在 Deviating TLS 中升高的 M3 / M4；M1a / M1b 已拆分为补充分析')
print(f'  EPAS1 GRN 靶基因直接参与这些代谢模块，支持 EPAS1 是调控 TLS 代谢差异的关键 TF')